# Preprocessing 04: Eye blinks and other artifacts

This notebook finds the stretches of the recording that contain **artifacts**, signals that don't come from the
brain, and labels them so later notebooks can leave them out. It **doesn't change the data**: it only adds labels
(MNE *annotations*: a start time, a length and a name).

| Label | What it marks | Found from | Step |
|---|---|---|---|
| `BAD_blink` | Eye blinks | the eye tracker's blinks (`ET_blink`), or the OPM sensors in front of the eyes if there is no eye tracking | 3, 4 |
| `BAD_saccade` | Large eye movements (optional, off by default) | the eye tracker's saccades (`ET_saccade`) | 5 |
| `BAD_muscle` | Bursts of muscle activity (jaw, neck, face) | high-frequency power on the OPM sensors (110-140 Hz), as in the FLUX pipeline | 6 |
| `BAD_jump` | Sudden jumps in single sensors (optional, off by default) | sample-to-sample changes larger than a threshold | 7 |
| `BAD_segment` | Anything else unusual: movement, sensor glitches, ... | a statistical outlier test (GESD), as in Harrison Ritz's mne-opm pipeline | 7 |

It does the same general check (`BAD_segment`, and `BAD_jump` if turned on) on the **empty-room recording** (Step 8).

**Notebook numbers.** The numbers start again in each folder of the repository. Here, **raw-files notebook 05**
is `01-explore-raw-files/05_Sync_eye_to_MEG.ipynb` (it syncs the eye tracker to the MEG), and **preprocessing
notebooks 01-03** are the notebooks before this one in this folder, `02-preprocessing/` (01: data and sensor
quality, 02: filter and downsample, 03: noise reduction).

**Why "BAD_".** MNE treats every label whose name starts with `bad` (upper or lower case) as data to leave out:
epochs that overlap one are dropped (`mne.Epochs(..., reject_by_annotation=True)`, the default), and ICA and
spectra skip those stretches. That is why the eye tracker's own labels from raw-files notebook 05 (`ET_blink`,
`ET_saccade`, `ET_missing`) don't start with `BAD_`: on their own they remove nothing.

**Blinks and ICA.** There are two ways to deal with blinks later: **drop** the trials that contain one (what
`BAD_blink` does when you epoch), or **remove** the blinks' field from the data with ICA and keep those trials. For
the second, a later ICA notebook has to stop MNE from leaving the blinks out, in two places:

1. **When fitting ICA**: `ica.fit(raw)` skips every `BAD...` stretch, so it would never see a blink and couldn't
   learn what one looks like. So it renames the label on a copy of the recording first, e.g.
   `raw_ica = raw.copy(); raw_ica.annotations.rename({"BAD_blink": "blink"})`, and fits ICA on that copy. The other
   `BAD_` labels (breaks, muscle, bad segments) are still skipped, which is what you want.
2. **When epoching**: after the blink components have been removed, the blinks are no longer artifacts, so it
   renames `BAD_blink` the same way before cutting epochs, and the trials with blinks are kept.

To find the blink components, ICA needs a signal that follows the blinks. With eye tracking, that is the blink
times (`ET_blink`): they come from a separate device, so they don't share anything with the MEG sensors, and ICA can
score its components against them (`ica.score_sources(raw, target=...)` takes any signal as long as the data, e.g.
1 during `ET_blink` and 0 elsewhere). Without eye tracking, it is the surrogate channel `OCC` that this notebook adds
(Step 4; `ica.find_bads_eog(raw, ch_name="OCC")`). The eye tracker's gaze and pupil channels are less useful for
this: raw-files notebook 05 fills the blinks in with straight lines, so the blinks barely show in them.

**One label per stretch of data.** The steps run from the most specific cause to the least specific: blinks, then
saccades, then muscle, then jumps, then everything else. Each step skips the data that an earlier step (or
preprocessing notebook 01's `BAD_break`) has already labelled, so no stretch gets two labels, and the totals in Step
9 add up. A burst of muscle activity during a blink is counted as part of the blink.

**It needs** the recording that preprocessing notebook 03 (noise reduction) saved, labelled `proc-denoise`, with its
sidecar files and its empty-room recording. The eye-tracking channels and labels were added by raw-files notebook 05
and carried along by preprocessing notebooks 02 and 03.

**It writes** a copy of the recording with the new labels, labelled `proc-annot`, to `derivatives/annotated/`, with
its sidecar files (Step 10), like preprocessing notebooks 02 and 03 do. The data are the same as preprocessing
notebook 03's; the labels are in its `_events.tsv`, so the next notebook gets them by loading it with
`read_raw_bids`. The empty room is saved the same way, with its own labels. For a recording **without eye
tracking**, the copy also gets one extra channel, `OCC`: the difference between two OPM sensors near the eyes, which
works like an EOG channel (Step 4), so a later ICA notebook can find the blinks with it.

You only edit Step 1. Then run the cells from top to bottom.

## Step 0: Load the Python packages

This cell loads the tools the notebook uses. You don't need to change anything.

`osl_gesd.py`, next to this notebook, holds the outlier test used in Step 7. It was copied from the osl-ephys
package, which is what the mne-opm pipeline uses, so that you don't need to install all of osl-ephys (see the top of
that file for its license and what was changed).

In [ ]:
# --- Standard Python tools (come with Python) ---
from pathlib import Path        # build and check file/folder paths
import json                     # read and write .json files
from datetime import datetime   # the date and time saved with the results

# --- Scientific Python tools ---
import numpy as np              # numbers and arrays (lists of numbers you can do math on)
import pandas as pd             # tables
import matplotlib.pyplot as plt # plotting
from scipy.signal import hilbert, detrend  # the envelope of a signal (Step 6); remove a straight-line trend (plots)
from scipy.fft import next_fast_len        # a length the Fourier transform handles quickly (Step 6)
from scipy.ndimage import label as label_runs   # numbers the separate runs of True in an array (Step 6)
from IPython.display import display        # shows a table nicely in the notebook

# --- MNE-Python: the MEG analysis package ---
import mne

# --- MNE-BIDS: reads BIDS files together with their sidecar files (Step 2) ---
import mne_bids

# --- The outlier test for Step 7, copied from osl-ephys (osl_gesd.py, next to this notebook) ---
from osl_gesd import bad_windows_channel_wise

# Only show MNE's warnings and errors (not every info message)
mne.set_log_level("WARNING")

# Draw plots inside the notebook
%matplotlib inline

print("MNE version:", mne.__version__)
print("MNE-BIDS version:", mne_bids.__version__)

## Step 1: Settings ✏️ (edit this cell)

Everything you normally need to change is in the cell below. Each setting is explained in the step that uses it.

In [ ]:
# ---------------------------------------------------------------------------
# WHICH RECORDING
# ---------------------------------------------------------------------------
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")  # top folder of your BIDS dataset
SUBJECT = "01"           # the part after "sub-"  (as text, in quotes)
SESSION = "01"           # the part after "ses-"; use None (no quotes) if you have no sessions
TASK = "oddballftcued"   # the part after "task-"
RUN = None               # the part after "run-", e.g. "01"; use None if there is no run label
DENOISE_DERIV = "denoise"    # the derivatives folder preprocessing notebook 03 saved the noise-reduced recording to
INPUT_PROC = "denoise"       # the proc- label of that recording (..._proc-denoise_meg.fif)
ANNOT_DERIV = "annotated"    # the derivatives folder this notebook saves the labelled recording to (Step 10)

# The eye tracker's labels from raw-files notebook 05 (only change these if you renamed them there)
ET_BLINK_ANNOT = "ET_blink"
ET_SACCADE_ANNOT = "ET_saccade"

# ---------------------------------------------------------------------------
# BLINKS (Steps 3 and 4)
# ---------------------------------------------------------------------------
BLINK_SOURCE = "eyetracker"     # where BAD_blink comes from: "eyetracker" (ET_blink) or "opm" (Step 4).
                                # Without eye-tracking data, "opm" is used either way.
BLINK_PAD_S = "auto"            # extra time added before and after each eye-tracker blink, in seconds,
                                # e.g. (0.1, 0.2); "auto" = what Step 3b measures in this recording
BLINK_PAD_THRESHOLD_SD = 3      # Step 3b: the blink's field counts as "still there" while it is more than
                                # this many standard deviations above the quiet baseline (this notebook's choice;
                                # Step 3b shows the pads for 2-5 SD)

# Blinks from the OPM sensors (Step 4)
SURROGATE_CHANNELS = "F9/F10"   # the two sensors whose difference stands in for an EOG channel (Step 4a):
                                # "F9/F10" = the Z sensors nearest the EEG positions F9 and F10, as in FLUX;
                                # or your own two, first minus second, e.g. ["3F[Z]", "2J[Z]"]
OPM_BLINK_THRESHOLD_FT = None   # how far a blink's peak must rise above the signal around it, in fT;
                                # None = MNE's automatic value (see Step 4b). FLUX used 3000 fT.
OPM_BLINK_BAND_HZ = (1, 10)     # the band the surrogate signal is filtered to before finding blinks (FLUX, MNE)
OPM_BLINK_WINDOW_S = "auto"     # the stretch labelled around each blink peak found this way, in seconds:
                                # "auto" = with eye tracking, the Step 3 pads moved to the peak (Step 4b);
                                # without, FLUX's (-0.25, 0.25). Or by hand, e.g. (-0.10, 1.07) (Step 4c suggests one)
ADD_SURROGATE_EOG = "auto"      # add the surrogate signal to the saved recording as an EOG channel named OCC
                                # (Step 10): "auto" = only when there are no eye-tracker blinks; or True / False

# ---------------------------------------------------------------------------
# SACCADES (Step 5)
# ---------------------------------------------------------------------------
MARK_SACCADES_BAD = False       # True: label large saccades BAD_saccade (False: only look at them)
SACCADE_BAD_MIN_DEG = 1.0       # with MARK_SACCADES_BAD: only saccades at least this large (degrees of visual
                                # angle; all saccades if their size can't be worked out)
SACCADE_PAD_S = (0.05, 0.05)    # with MARK_SACCADES_BAD: extra time before and after each saccade (seconds)

# ---------------------------------------------------------------------------
# MUSCLE (Step 6)
# ---------------------------------------------------------------------------
MUSCLE_BAND_HZ = (110, 140)     # the frequency band muscle activity is measured in (FLUX)
MUSCLE_NOTCH_HZ = [120]         # power-line harmonics inside that band, removed from the copy used for detection
                                # only (60 Hz power: 120 Hz is inside 110-140 Hz). [] = none
MUSCLE_THRESHOLD = 3.0          # z-score above which data count as muscle (FLUX: 3; MNE's default: 4)
MUSCLE_MIN_LENGTH_GOOD_S = 0.1  # good stretches shorter than this between two muscle stretches are joined to them

# ---------------------------------------------------------------------------
# JUMPS AND OTHER BAD SEGMENTS (Step 7) AND THE EMPTY ROOM (Step 8)
# ---------------------------------------------------------------------------
JUMP_THRESHOLD_PT = None        # label a jump when one sensor changes by more than this between two samples,
                                # in pT (picotesla); None = off. See the table in Step 7a before turning it on.
SEGMENT_BAND_HZ = (1, 40)       # the band the data are filtered to for the general check (a copy only):
                                # usually the band you will analyse
SEGMENT_WINDOW_S = 0.5          # length of the windows the data are cut into (mne-opm stage 2: 0.5 s)
SEGMENT_CHANNEL_THRESHOLD = 0.05   # a window is bad when at least this share of the sensors is unusual in it
                                   # (mne-opm stage 2: 0.05, i.e. 5 %)
SEGMENT_SIGNIFICANCE = 0.05     # significance level of the outlier test (osl-ephys' default)
PROCESS_EMPTY_ROOM = True       # True: run the Step 7 checks on the empty-room recording too (Step 8)
EMPTY_ROOM_CHANNEL_THRESHOLD = 0.30   # the share of sensors for the empty room (mne-opm stage 2: 0.30)

# ---------------------------------------------------------------------------
# HOW MANY TRIALS WOULD BE LOST (Step 9c)
# ---------------------------------------------------------------------------
# The events your epochs will be cut around, each with the epoch you plan to use (start, end), in seconds
# around the event; add more events as "name": (start, end). In the sample task each trial is cut around the
# deviant and includes the standard just before it, which came 1.11-1.32 s earlier: the epoch starts 0.2 s before
# the latest of those standards (-1.32 - 0.2 = -1.52 s).
PREVIEW_EPOCHS = {"deviant": (-1.52, 0.8)}

# ---------------------------------------------------------------------------
# BROWSER AND SAVING
# ---------------------------------------------------------------------------
SHOW_BROWSER = False            # True: open MNE's interactive data browser in Step 9e
BROWSE_DURATION_S = 10          # seconds shown at a time in the browser
OVERWRITE = True                # True: replace the files saved by an earlier run of this notebook

## Step 2: Find the file and load the data

This cell works out the path of preprocessing notebook 03's recording from your settings, checks it exists, and reads it with
`read_raw_bids`, together with its sidecar files: the bad channels come from its `_channels.tsv`, and the events
(task events, eye-tracking labels and preprocessing notebook 01's breaks) from its `_events.tsv`, as annotations.

`raw.load_data()` reads all the data into memory, which the detection steps need. A long recording takes a few GB
and can take a minute or two to load. The steps below are written to work through the data one sensor at a time
where they can, so they don't need a second full copy in memory.

You will see the warning "participants.tsv file not found": it is expected and harmless (see preprocessing notebook 02, Step 2).

Then the cell lists the labels the recording already has, and what eye-tracking information is available.

In [ ]:
BIDS_ROOT = Path(BIDS_ROOT)

if "[yourpath]" in str(BIDS_ROOT):
    raise ValueError("BIDS_ROOT still contains the [yourpath] placeholder. Set it in Step 1.")

# ---- The BIDS-style name of this recording, e.g. sub-01_ses-01_task-oddball ------------------
sub_ses = [f"sub-{SUBJECT}"] + ([f"ses-{SESSION}"] if SESSION else [])   # folder levels
name_parts = sub_ses + [f"task-{TASK}"] + ([f"run-{RUN}"] if RUN else [])
base = "_".join(name_parts)        # the start of every file name for this recording
sub_dir = Path(*sub_ses)           # e.g. sub-01/ses-01
input_base = f"{base}_proc-{INPUT_PROC}"   # the start of the input file names, e.g. ..._proc-denoise

# ---- Preprocessing notebook 03's recording --------------------------------------------------------------------
denoise_root = BIDS_ROOT / "derivatives" / DENOISE_DERIV
bids_path = mne_bids.BIDSPath(root=denoise_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                              processing=INPUT_PROC, datatype="meg", suffix="meg", extension=".fif")
# A recording over 2 GB is saved in parts (..._split-01_meg.fif, ...): open the first, MNE finds the others
if not bids_path.fpath.is_file() and bids_path.copy().update(split="01").fpath.is_file():
    bids_path.update(split="01")
meg_file = bids_path.fpath
meg_dir = meg_file.parent
if not meg_file.is_file():
    found = sorted(meg_dir.glob("*_meg.fif")) if meg_dir.is_dir() else []
    raise FileNotFoundError(
        f"Recording not found:\n    {meg_file}\nHas preprocessing notebook 03 been run for this session? FIF files in that "
        "folder: " + (", ".join(p.name for p in found) or "none") + "\n(To run this notebook on preprocessing notebook 02's "
        'output instead, set DENOISE_DERIV = "filtered" and INPUT_PROC = "filt" in Step 1.)')

raw = mne_bids.read_raw_bids(bids_path)   # "participants.tsv file not found": expected (see above)
raw.load_data()
print("Loaded:", meg_file)
if len(raw.filenames) > 1:
    print(f"(the recording is split into {len(raw.filenames)} files; MNE read them all)")

# read_raw_bids loses the coil type of the eye-tracker channels (see preprocessing notebook 01, Step 2b);
# the FIF file's own header still has it, so copy it back. No file is changed.
header = mne.io.read_info(meg_file, verbose=False)
for ch_in_file in header["chs"]:
    if ch_in_file["ch_name"] in raw.ch_names:
        ch = raw.info["chs"][raw.ch_names.index(ch_in_file["ch_name"])]
        if ch["kind"] == ch_in_file["kind"]:
            ch["coil_type"] = ch_in_file["coil_type"]

SFREQ = raw.info["sfreq"]
print(raw)
print(f"Sampling rate {SFREQ:g} Hz, {raw.times[-1] / 60:.1f} min; bad channels: {raw.info['bads'] or 'none'}")

# ---- The labels this notebook adds: they shouldn't be in the input yet ------------------------------
OUR_LABELS = ["BAD_blink", "BAD_saccade", "BAD_muscle", "BAD_jump", "BAD_segment", "OPM_blink"]
already = [d for d in OUR_LABELS if d in set(raw.annotations.description)]
if already:
    print(f"\nThe input already has {already} (from an earlier run?). They are removed here and found again.")
    raw.annotations.delete(np.flatnonzero(np.isin(raw.annotations.description, already)))

# ---- The labels the recording already has --------------------------------------------------------
existing = (pd.DataFrame({"label": raw.annotations.description, "duration_s": raw.annotations.duration})
            .groupby("label").agg(count=("label", "size"), total_s=("duration_s", "sum")).round(1))
print("\nLabels (annotations) in the recording:")
display(existing)

# ---- What eye-tracking information there is --------------------------------------------------------
labels_present = set(raw.annotations.description)
HAS_ET_BLINKS = ET_BLINK_ANNOT in labels_present
HAS_ET_SACCADES = ET_SACCADE_ANNOT in labels_present
HAS_GAZE = all(ch in raw.ch_names for ch in ("ET_xpos", "ET_ypos"))
print(f"Eye-tracker blinks ({ET_BLINK_ANNOT}): {HAS_ET_BLINKS};  saccades ({ET_SACCADE_ANNOT}): {HAS_ET_SACCADES};  "
      f"gaze channels (ET_xpos, ET_ypos): {HAS_GAZE}")

if BLINK_SOURCE not in ("eyetracker", "opm"):
    raise ValueError(f'BLINK_SOURCE must be "eyetracker" or "opm", not {BLINK_SOURCE!r} (Step 1).')
blink_source = BLINK_SOURCE
if BLINK_SOURCE == "eyetracker" and not HAS_ET_BLINKS:
    blink_source = "opm"
    print(f"\nNo {ET_BLINK_ANNOT} labels in this recording, so BAD_blink will come from the OPM sensors (Step 4).")
print(f"BAD_blink will come from: {'the eye tracker (Step 3)' if blink_source == 'eyetracker' else 'the OPM sensors (Step 4)'}")

if ADD_SURROGATE_EOG not in ("auto", True, False):
    raise ValueError(f'ADD_SURROGATE_EOG must be "auto", True or False, not {ADD_SURROGATE_EOG!r} (Step 1).')
add_occ = (not HAS_ET_BLINKS) if ADD_SURROGATE_EOG == "auto" else ADD_SURROGATE_EOG
print(f"Surrogate channel OCC in the saved recording (Step 10): {'yes' if add_occ else 'no'}"
      + (" (auto: " + ("no eye-tracker blinks)" if not HAS_ET_BLINKS else "the eye tracker's blinks are there)")
         if ADD_SURROGATE_EOG == "auto" else " (set in Step 1)"))

### Step 2b: Tools used in the rest of the notebook

This cell only defines a few small functions (no output). They work with **spans**: stretches of time given by
their start and end, in seconds from the first sample of the recording, the same times as in the plots.

- `get_spans(rec, name)`: the spans of all annotations with that name.
- `set_spans(rec, name, starts, ends)`: replaces all annotations with that name by these spans (so running a cell
  twice doesn't add them twice).
- `merge_spans(...)`: joins spans that overlap or touch; `subtract_spans(...)`: cuts away the parts already covered
  by other spans (used to keep one label per stretch).
- `bad_spans(rec)`: the stretches covered by any `BAD...` annotation, i.e. the data MNE would leave out.
- `plot_window(...)`: plots a few seconds of chosen sensors (and the eye tracker, if there is one), with the labels
  shaded. Each trace's slow drift (a straight line through the window) is removed for the plot only.

**A note on times.** MNE counts annotation times from the recording's measurement date, which is `raw.first_time`
earlier than the first stored sample; the functions take care of that difference (`span_shift`).

In [ ]:
def span_shift(rec):
    # Annotation onsets count from the measurement date, which is first_time before the first sample (when set)
    return rec.first_time if rec.annotations.orig_time is not None else 0.0


def rec_length_s(rec):
    return rec.n_times / rec.info["sfreq"]


def get_spans(rec, name):
    # (starts, ends) of all annotations called `name`, in seconds from the first sample, sorted by start
    a = rec.annotations
    sel = a.description == name
    starts = a.onset[sel] - span_shift(rec)
    ends = starts + a.duration[sel]
    order = np.argsort(starts)
    return starts[order], ends[order]


def merge_spans(starts, ends):
    # Join spans that overlap or touch; returns sorted (starts, ends)
    starts, ends = np.asarray(starts, float), np.asarray(ends, float)
    if len(starts) == 0:
        return np.array([]), np.array([])
    order = np.argsort(starts)
    merged_s, merged_e = [starts[order[0]]], [ends[order[0]]]
    for s, e in zip(starts[order[1:]], ends[order[1:]]):
        if s <= merged_e[-1]:
            merged_e[-1] = max(merged_e[-1], e)
        else:
            merged_s.append(s)
            merged_e.append(e)
    return np.array(merged_s), np.array(merged_e)


def subtract_spans(starts, ends, cut_starts, cut_ends):
    # The parts of the spans (starts, ends) that are NOT covered by the spans (cut_starts, cut_ends)
    starts, ends = merge_spans(starts, ends)
    cut_starts, cut_ends = merge_spans(cut_starts, cut_ends)
    out_s, out_e = [], []
    for s, e in zip(starts, ends):
        pos = s
        for cs, ce in zip(cut_starts, cut_ends):
            if ce <= pos or cs >= e:
                continue
            if cs > pos:
                out_s.append(pos)
                out_e.append(cs)
            pos = max(pos, ce)
            if pos >= e:
                break
        if pos < e:
            out_s.append(pos)
            out_e.append(e)
    return np.array(out_s), np.array(out_e)


def total_s(starts, ends):
    # Total time covered (overlaps counted once)
    s, e = merge_spans(starts, ends)
    return float(np.sum(e - s))


def overlap_s(spans_a, spans_b):
    # Total time covered by both span lists
    a_s, a_e = merge_spans(*spans_a)
    b_s, b_e = merge_spans(*spans_b)
    total = 0.0
    for s, e in zip(a_s, a_e):
        total += np.sum(np.clip(np.minimum(b_e, e) - np.maximum(b_s, s), 0, None))
    return float(total)


def set_spans(rec, name, starts, ends):
    # Replace all annotations called `name` with these spans (clipped to the recording)
    length = rec_length_s(rec)
    starts = np.clip(np.asarray(starts, float), 0, length)
    ends = np.clip(np.asarray(ends, float), 0, length)
    keep = ends > starts
    rec.annotations.delete(np.flatnonzero(rec.annotations.description == name))
    if keep.any():
        rec.annotations.append(starts[keep] + span_shift(rec), ends[keep] - starts[keep], name)


def bad_spans(rec, exclude=()):
    # The stretches covered by any annotation starting with "bad" (upper or lower case), merged
    a = rec.annotations
    is_bad = np.array([d.lower().startswith("bad") and d not in exclude for d in a.description], bool)
    starts = a.onset[is_bad] - span_shift(rec)
    return merge_spans(starts, starts + a.duration[is_bad])


def spans_to_mask(rec, starts, ends):
    # True for every sample inside one of the spans
    mask = np.zeros(rec.n_times, bool)
    sf = rec.info["sfreq"]
    for s, e in zip(starts, ends):
        mask[max(int(np.floor(s * sf)), 0):max(int(np.ceil(e * sf)), 0)] = True
    return mask


def mask_to_spans(mask, sfreq):
    # The runs of True in a sample mask, as (starts, ends) in seconds
    padded = np.r_[False, mask, False].astype(int)
    edges = np.flatnonzero(np.diff(padded))
    return edges[0::2] / sfreq, edges[1::2] / sfreq


def in_spans(times, starts, ends):
    # For each time: is it inside one of the spans?
    times = np.asarray(times, float)
    s, e = merge_spans(starts, ends)
    if len(s) == 0:
        return np.zeros(times.shape, bool)
    k = np.searchsorted(s, times, side="right") - 1
    return (k >= 0) & (times < e[np.clip(k, 0, None)])


# The good magnetometers (OPM sensors) and the ones measuring the Z direction (used by FLUX for blinks)
mag_names = [raw.ch_names[i] for i in mne.pick_types(raw.info, meg="mag", ref_meg=False, exclude="bads")]
z_names = [n for n in mag_names if n.endswith("[Z]")] or mag_names

# The breaks from preprocessing notebook 01 and the time outside them (the "usable" time used for rates and percentages)
break_spans = get_spans(raw, "BAD_break")
usable_s = rec_length_s(raw) - total_s(*break_spans)
print(f"{len(mag_names)} good OPM channels ({len(z_names)} Z channels); "
      f"{usable_s / 60:.1f} of {rec_length_s(raw) / 60:.1f} min outside breaks")

# Colours for the labels in the plots
LABEL_COLORS = {"BAD_break": "0.75", "ET_blink": "tab:blue", "BAD_blink": "tab:red", "OPM_blink": "tab:orange",
                "ET_saccade": "tab:green", "BAD_saccade": "tab:olive", "BAD_muscle": "tab:purple",
                "BAD_jump": "tab:brown", "BAD_segment": "tab:pink"}


def plot_window(rec, t0, t1, channels, labels, title, gaze=True):
    # Plot t0-t1 s of `channels` (fT), the eye tracker's vertical gaze and pupil if present, labels shaded
    sf = rec.info["sfreq"]
    start, stop = max(int(t0 * sf), 0), min(int(t1 * sf), rec.n_times)
    t = np.arange(start, stop) / sf
    show_gaze = gaze and all(ch in rec.ch_names for ch in ("ET_ypos", "ET_pupil"))
    fig, axes = plt.subplots(2 if show_gaze else 1, 1, figsize=(13, 5.5 if show_gaze else 3.8), sharex=True,
                             squeeze=False, gridspec_kw={"height_ratios": [3, 1] if show_gaze else [1]})
    ax = axes[0, 0]
    data = rec.get_data(picks=channels, start=start, stop=stop) * 1e15        # T -> fT
    data = detrend(data, axis=1)              # remove each trace's slow drift (a straight line), for display only
    step = np.percentile(np.ptp(data, axis=1), 75) or 1.0
    for i, ch in enumerate(channels):
        ax.plot(t, data[i] - i * step, lw=0.7, color="k")
    ax.set_yticks([-i * step for i in range(len(channels))], channels)
    ax.set_ylabel(f"OPM sensors (one tick = {step:.0f} fT)")
    if show_gaze:
        gz = rec.get_data(picks=["ET_ypos", "ET_pupil"], start=start, stop=stop)
        gz = (gz - gz.mean(axis=1, keepdims=True)) / (gz.std(axis=1, keepdims=True) + 1e-12)
        axes[1, 0].plot(t, gz[0], label="vertical gaze (ET_ypos)", color="tab:cyan")
        axes[1, 0].plot(t, gz[1], label="pupil (ET_pupil)", color="tab:gray")
        axes[1, 0].set_ylabel("eye tracker\n(z-scored)")
        axes[1, 0].legend(loc="upper right", fontsize=8)
    handles = []
    for name in labels:
        s, e = get_spans(rec, name)
        sel = (e > t0) & (s < t1)
        for a in axes[:, 0]:
            for si, ei in zip(s[sel], e[sel]):
                a.axvspan(max(si, t0), min(ei, t1), color=LABEL_COLORS.get(name, "0.5"), alpha=0.25, lw=0)
        handles.append(plt.Rectangle((0, 0), 1, 1, color=LABEL_COLORS.get(name, "0.5"), alpha=0.4, label=name))
    axes[0, 0].legend(handles=handles, loc="upper right", fontsize=8, ncol=len(handles) or 1)
    axes[-1, 0].set_xlabel("Time (s from the start of the recording)")
    axes[0, 0].set_xlim(t0, t1)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## Step 3: Blinks from the eye tracker

Raw-files notebook 05 put the eye tracker's blinks on the MEG timeline as `ET_blink`. EyeLink reports each blink inside a
saccade, because the eyelid moving over the pupil looks like a fast eye movement, and an `ET_blink` span covers that
whole saccade: from the lid starting to close to it being open again (see raw-files notebook 05, Step 13).

This step turns them into `BAD_blink`, in three parts:

- **3a** looks at the blinks: how many, how long, and when.
- **3b** measures, in this recording, how long the blinks' magnetic field lasts **before and after** the
  eye tracker's blink span. The eyelid and the eye move before the eye tracker loses the pupil and after it finds it
  again, so the field can start earlier and end later than `ET_blink`. Step 3b measures by how much, and that
  becomes the extra time added around each blink (`BLINK_PAD_S = "auto"` in Step 1).
- **3c** makes `BAD_blink`: each `ET_blink`, plus that extra time.

If the recording has no `ET_blink` labels (no eye tracking), this step is skipped and Step 4 finds the blinks
from the OPM sensors.

### Step 3a: The eye tracker's blinks

The table and the histogram only use the blinks **outside the breaks**: any blink that overlaps a break, even in
part, is left out, here and in the rest of the notebook. During breaks people may close their eyes for several
seconds, which the eye tracker also reports as `ET_blink` (up to 9.8 s in the sample recording). Those spans
are not blinks in the usual sense, and they are already inside `BAD_break`.

Outside the breaks, a few blinks can still be longer than 1 s: the cell lists when. In the sample recording the longest
of them start during the 500 ms BLINK screen at the end of a trial, where blinking was allowed. They
stay in: they still are periods with an eye-closure field that `BAD_blink` should cover.

In [ ]:
if not HAS_ET_BLINKS:
    print(f"No {ET_BLINK_ANNOT} labels: Step 3 is skipped (blinks come from the OPM sensors, Step 4).")
else:
    et_blink = get_spans(raw, ET_BLINK_ANNOT)
    et_dur_ms = (et_blink[1] - et_blink[0]) * 1000
    et_mid = (et_blink[0] + et_blink[1]) / 2
    # A blink counts as "in a break" if any part of it overlaps a break. Then a long eye closure that starts just
    # before a break (in the sample recording: one of 5 s) is left out too, not only those inside it.
    et_in_break = np.array([overlap_s(([s], [e]), break_spans) > 0 for s, e in zip(*et_blink)], bool)
    print(f"{len(et_blink[0])} {ET_BLINK_ANNOT} spans, {et_in_break.sum()} of them in or touching breaks (left out below). "
          f"Outside breaks: {(~et_in_break).sum() / (usable_s / 60):.1f} blinks per minute, "
          f"{total_s(et_blink[0][~et_in_break], et_blink[1][~et_in_break]):.0f} s in all "
          f"({100 * total_s(et_blink[0][~et_in_break], et_blink[1][~et_in_break]) / usable_s:.1f} % of the time).")
    display(pd.Series(et_dur_ms[~et_in_break], name=f"{ET_BLINK_ANNOT} duration outside breaks (ms)")
            .describe().round(0).to_frame().T)
    long_blinks = (~et_in_break) & (et_dur_ms > 1000)
    print(f"Longer than 1 s (outside breaks): {long_blinks.sum()}" + (", at " + ", ".join(f"{t:.0f}" for t in
          et_blink[0][long_blinks][:10]) + (" ..." if long_blinks.sum() > 10 else "") + " s" if long_blinks.any() else ""))

    fig, (ax_h, ax_t) = plt.subplots(1, 2, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 2]})
    ax_h.hist(et_dur_ms[~et_in_break], bins=40, color="tab:blue")
    ax_h.set(xlabel="Duration (ms)", ylabel="Blinks", title=f"{ET_BLINK_ANNOT} durations (outside breaks)")
    for s, e in zip(*break_spans):
        ax_t.axvspan(s, e, color=LABEL_COLORS["BAD_break"], alpha=0.5, lw=0)
    ax_t.eventplot(et_mid, lineoffsets=0.5, linelengths=0.8, color="tab:blue")
    ax_t.set(xlim=(0, rec_length_s(raw)), yticks=[], xlabel="Time (s)",
             title=f"When the blinks happened (grey: breaks, BAD_break)")
    fig.tight_layout()
    plt.show()

### Step 3b: How long does a blink's field last?

This cell averages the OPM signals around the eye tracker's blinks, twice:

- **locked to the blink start** (`ET_blink` onset, time 0), to see how early the field starts;
- **locked to the blink end** (the end of the `ET_blink` span, time 0), to see how long it lasts afterwards.

Averaging hundreds of blinks removes most of the brain activity and noise, and leaves the field the blinks have in
common. The averages are smoothed (low-pass 40 Hz) and corrected to a quiet stretch: 0.5-0.3 s before the blink
start, and 0.4-0.6 s after the blink end. Blinks during breaks are left out.

The **global field power** (GFP: the root mean square over all sensors at each moment) sums up the field's
strength in one curve. The field counts as present while the GFP is more than `BLINK_PAD_THRESHOLD_SD` (3) standard
deviations above its value in the quiet stretch (the dashed line). The 3 is a choice made for this notebook, not a
published value: a level that the GFP rarely reaches by chance in the quiet stretch, so that crossing it means the
blink's field is there. A lower value gives longer pads, a higher one shorter pads; the small table under the plots
shows the pads for 2 to 5 SD, so you can see how much the choice matters in your recording. That gives the extra time needed before and after
the span:

- **pad before** = from the first moment after the quiet stretch that the GFP rises above the line, to the
  `ET_blink` start;
- **pad after** = from the `ET_blink` end to the last moment the GFP is above the line before the quiet stretch.
  The field can dip below the line and come back (e.g. when the eye settles after the lid opens); the pad covers
  that too.

If the GFP is below the line at time 0 and stays there, the pad is 0. The pads can't be longer than the time between
time 0 and the quiet stretch (0.3 s before, 0.4 s after).

The table lists the sensors with the largest blink field. They are used to show blinks in the plots below. The map shows the field at its peak on the
Z channels, seen from above with the nose at the top.

**What to look for:** a clear deflection on the front sensors starting near time 0 in the left plot and fading out
after time 0 in the right plot. The pads should be a few tens to a few hundreds of milliseconds. If a pad reaches its
maximum, the field may last even longer; if the curves are noisy (few blinks), the pads are uncertain. In both cases,
set `BLINK_PAD_S` in Step 1 by hand instead.

In [ ]:
blink_pad_measured = None    # (before, after) in s, measured here
BLINK_SENSORS = []           # the sensors with the largest blink field


def locked_average(times_s, tmin, tmax, baseline):
    # Average the good OPM channels around the given times (seconds from the first sample); epochs that overlap
    # a BAD annotation (here: the breaks) are left out. The average is smoothed with a 40 Hz low-pass filter.
    samples = np.unique(np.round(np.asarray(times_s) * SFREQ).astype(int)) + raw.first_samp
    events = np.column_stack([samples, np.zeros_like(samples), np.ones_like(samples)])
    epochs = mne.Epochs(raw, events, tmin=tmin, tmax=tmax, picks=mag_names, baseline=baseline,
                        reject_by_annotation=True, preload=False, verbose="error")
    evoked = epochs.average()    # reads the epochs one at a time, so this needs little memory
    return evoked.filter(None, 40.0, method="iir", verbose="error")


def gfp(evoked):
    return np.sqrt(np.mean(evoked.data ** 2, axis=0))


def measure_pad(evoked, quiet, direction):
    # direction = -1: how long before time 0 the GFP first rises above the threshold (after the quiet stretch)
    # direction = +1: how long after time 0 it is last above the threshold (before the quiet stretch)
    g, t = gfp(evoked), evoked.times
    in_quiet = (t >= quiet[0]) & (t <= quiet[1])
    thr = g[in_quiet].mean() + BLINK_PAD_THRESHOLD_SD * g[in_quiet].std()
    if direction < 0:
        between = (t > quiet[1]) & (t <= 0)
        above = np.flatnonzero(between & (g > thr))
        return (-t[above[0]] if len(above) else 0.0), thr
    between = (t >= 0) & (t < quiet[0])
    above = np.flatnonzero(between & (g > thr))
    return (t[above[-1]] + 1 / evoked.info["sfreq"] if len(above) else 0.0), thr


if not HAS_ET_BLINKS:
    print("No eye-tracker blinks: Step 3b is skipped.")
else:
    use = ~et_in_break
    print("Averaging the blinks (this reads each blink's data once; it can take a minute) ...")
    ev_start = locked_average(et_blink[0][use], -0.5, 0.6, (-0.5, -0.3))
    ev_end = locked_average(et_blink[1][use], -0.6, 0.6, (0.4, 0.6))
    pad_before, thr_start = measure_pad(ev_start, (-0.5, -0.3), -1)
    pad_after, thr_end = measure_pad(ev_end, (0.4, 0.6), +1)
    blink_pad_measured = (round(pad_before, 3), round(pad_after, 3))
    print(f"Blinks averaged: {ev_start.nave} (start-locked), {ev_end.nave} (end-locked)")
    print(f"Measured pads: {pad_before * 1000:.0f} ms before the ET_blink start, {pad_after * 1000:.0f} ms after its end")

    # ---- The sensors with the largest blink field: at the GFP peak after the blink start ---------------
    after = (ev_start.times >= 0) & (ev_start.times <= 0.5)
    t_peak = ev_start.times[after][np.argmax(gfp(ev_start)[after])]
    amp = ev_start.data[:, np.argmin(np.abs(ev_start.times - t_peak))] * 1e15     # fT, one value per channel
    ranking = pd.DataFrame({"channel": ev_start.ch_names, "field at the peak (fT)": amp.round(0)})
    ranking = ranking.reindex(ranking["field at the peak (fT)"].abs().sort_values(ascending=False).index)
    BLINK_SENSORS = list(ranking["channel"].head(6))
    print(f"\nGFP peak {t_peak * 1000:.0f} ms after the blink start. Largest blink fields:")
    display(ranking.head(8).reset_index(drop=True))

    # ---- Plots: the averages and the GFP with the threshold and the measured pads ------------------------
    fig, axes = plt.subplots(2, 2, figsize=(13, 6), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
    for col, (ev, thr, label, pad, sign) in enumerate([(ev_start, thr_start, "blink start", pad_before, -1),
                                                       (ev_end, thr_end, "blink end", pad_after, +1)]):
        axes[0, col].plot(ev.times, ev.data.T * 1e15, lw=0.5, color="k", alpha=0.4)
        axes[0, col].set(ylabel="Field (fT)", title=f"Average locked to the {label} (n = {ev.nave})")
        axes[1, col].plot(ev.times, gfp(ev) * 1e15, color="tab:blue", label="GFP")
        axes[1, col].axhline(thr * 1e15, color="tab:red", ls="--", lw=1, label=f"threshold ({BLINK_PAD_THRESHOLD_SD} SD)")
        axes[1, col].axvspan(min(0, sign * pad), max(0, sign * pad), color="tab:red", alpha=0.2, label=f"pad {pad * 1000:.0f} ms")
        axes[1, col].set(xlabel=f"Time from the {label} (s)", ylabel="GFP (fT)")
        axes[1, col].legend(fontsize=8, loc="upper right")
        for a in axes[:, col]:
            a.axvline(0, color="tab:blue", lw=1)
    fig.tight_layout()
    plt.show()

    # ---- How much the threshold matters: the pads for 2 to 5 SD (the one used is BLINK_PAD_THRESHOLD_SD) --------
    sd_used = BLINK_PAD_THRESHOLD_SD
    sd_rows = []
    for sd in [2, 3, 4, 5]:
        BLINK_PAD_THRESHOLD_SD = sd           # measure_pad reads this setting
        sd_rows.append({"threshold (SD)": sd, "pad before (ms)": round(1000 * measure_pad(ev_start, (-0.5, -0.3), -1)[0]),
                        "pad after (ms)": round(1000 * measure_pad(ev_end, (0.4, 0.6), +1)[0])})
    BLINK_PAD_THRESHOLD_SD = sd_used          # back to the value from Step 1
    print(f"Pads for other thresholds (used: {sd_used} SD):")
    display(pd.DataFrame(sd_rows).set_index("threshold (SD)"))

    # ---- Map of the field at the peak, Z channels only ------------------------------------------------
    try:
        ev_start.copy().pick(z_names).plot_topomap(times=[t_peak], ch_type="mag", units="fT", scalings=1e15,
                                                   size=3, time_format="%0.3f s after the blink start")
    except Exception as err:      # the map needs sensor positions; skip it if they are missing
        print("(The field map could not be drawn:", err, ")")

### Step 3c: Make `BAD_blink`

Each `ET_blink` span, plus the pad before and after it (`BLINK_PAD_S`: the measured values with `"auto"`, or the
values you set in Step 1), becomes a `BAD_blink` span. Spans that then overlap are joined, and the parts that fall
in a break (already `BAD_break`) are cut away, so each stretch keeps a single label.

The plot shows one typical blink (the one with the median duration): the sensors with the largest blink field, the eye tracker's vertical gaze and pupil size
below (both drop out during the blink, where raw-files notebook 05 filled them in with straight lines), `ET_blink` in blue and
`BAD_blink` in red.

In [ ]:
if BLINK_PAD_S == "auto":
    blink_pad = blink_pad_measured if blink_pad_measured is not None else (0.0, 0.0)
else:
    blink_pad = tuple(float(x) for x in BLINK_PAD_S)

if blink_source != "eyetracker":
    print("BLINK_SOURCE is 'opm' (or there is no eye tracking): BAD_blink is made from the OPM sensors in Step 4d.")
else:
    s, e = merge_spans(et_blink[0] - blink_pad[0], et_blink[1] + blink_pad[1])
    s, e = subtract_spans(s, e, *bad_spans(raw, exclude=("BAD_blink",)))   # not inside breaks
    set_spans(raw, "BAD_blink", s, e)
    print(f"BAD_blink: {len(s)} spans, {total_s(s, e):.0f} s = {100 * total_s(s, e) / usable_s:.1f} % of the time "
          f"outside breaks (ET_blink plus {blink_pad[0] * 1000:.0f} ms before and {blink_pad[1] * 1000:.0f} ms after; "
          f"{'measured in Step 3b' if BLINK_PAD_S == 'auto' else 'set in Step 1'})")

    # One typical blink outside the breaks: the one whose duration is closest to the median
    candidates = np.flatnonzero(~et_in_break)
    k = candidates[np.argmin(np.abs(et_dur_ms[candidates] - np.median(et_dur_ms[candidates])))]
    centre = et_mid[k]
    plot_window(raw, centre - 1.5, centre + 1.5, BLINK_SENSORS or z_names[:6], [ET_BLINK_ANNOT, "BAD_blink", "BAD_break"],
                f"A typical blink, at {centre:.1f} s: {ET_BLINK_ANNOT} (blue) and BAD_blink (red)")

## Step 4: Blinks from the OPM sensors (for recordings without eye tracking)

Without an eye tracker, blinks can be found from the OPM sensors themselves, the way the FLUX pipeline does it
([FLUX, ArtefactAnnotation](https://github.com/FLUX-pipeline/Cerca/blob/main/ArtefactAnnotation.ipynb)): the
difference between two sensors in front of the eyes works like an EOG channel (a *surrogate* ocular channel, which
FLUX calls `OCC`). With an eye tracker, this step is a check of how well that works: Step 4c compares the blinks it
finds with the eye tracker's. That tells you how far to trust it for recordings without eye tracking, and what
threshold to use for them.

- **4a** chooses the two sensors.
- **4b** builds the surrogate signal and finds the blinks in it, labelled `OPM_blink`.
- **4c** compares `OPM_blink` with the eye tracker's blinks (only with eye tracking).
- **4d** uses `OPM_blink` as `BAD_blink` if `BLINK_SOURCE = "opm"` or there is no eye tracking. Otherwise `OPM_blink`
  is only used for the comparison and is not saved.

Without eye tracking, Step 10 also saves the surrogate signal itself, as a channel named `OCC` (see
`ADD_SURROGATE_EOG` in Step 1).

### Step 4a: The two sensors

FLUX subtracts a right front sensor from a left front sensor, both Z direction, in the helmet positions FLUX labels
F9 and F10. In the standard EEG layout, F9 and F10 are low on the left and right sides of the forehead, towards the
temples (in the row below F7 and F8). This notebook does the same: it takes the Z sensors nearest to the standard
positions F9 and F10 on MNE's template head (`colin27_1005`, called `standard_1005` before MNE 1.13), compared with
the sensor positions in head coordinates. The positions come from each recording's head position, so if a head sits
differently in the helmet, the nearest sensors can be different ones.

During a blink the two sensors see fields of opposite sign (in the sample recording 3F[Z] positive and 2J[Z]
negative), so subtracting one from the other adds up their blink fields. To use two other sensors, name them in
`SURROGATE_CHANNELS` in Step 1.

The map is the same as in preprocessing notebook 01 (Step 5): the Z channel of each sensor, seen from above with the
nose at the top, with its name; bad sensors are red. The pair is marked on it: ▲ is the first sensor and ▼ the second
(the surrogate signal is first minus second).

In [ ]:
def head_positions(rec, names):
    # Sensor positions (m) in head coordinates: x to the right ear, y to the nose, z up
    pos = np.array([rec.info["chs"][rec.ch_names.index(n)]["loc"][:3] for n in names])
    if rec.info["dev_head_t"] is not None:
        pos = mne.transforms.apply_trans(rec.info["dev_head_t"], pos)
    return pos


def surrogate_signal(rec, pair):
    # The first sensor minus the second, in T
    return rec.get_data(picks=[pair[0]])[0] - rec.get_data(picks=[pair[1]])[0]


def surrogate_raw(rec, pair):
    # A one-channel recording holding the surrogate signal as an EOG channel named OCC, with the breaks
    info = mne.create_info(["OCC"], rec.info["sfreq"], "eog")
    occ = mne.io.RawArray(surrogate_signal(rec, pair)[np.newaxis], info, first_samp=rec.first_samp, verbose="error")
    occ.set_meas_date(rec.info["meas_date"])
    occ.set_annotations(rec.annotations[rec.annotations.description == "BAD_break"])
    return occ


def opm_blink_peaks(occ, thresh_T):
    # Times (s from the first sample) of the blink peaks find_eog_events finds; the breaks are left out
    events = mne.preprocessing.find_eog_events(occ, ch_name="OCC", thresh=thresh_T, l_freq=OPM_BLINK_BAND_HZ[0],
                                               h_freq=OPM_BLINK_BAND_HZ[1], reject_by_annotation=True, verbose="error")
    return (events[:, 0] - occ.first_samp) / occ.info["sfreq"]


def filtered_surrogate(occ):
    # The surrogate signal filtered the way find_eog_events filters it (for plots and MNE's automatic threshold)
    return mne.filter.filter_data(occ.get_data()[0], occ.info["sfreq"], OPM_BLINK_BAND_HZ[0], OPM_BLINK_BAND_HZ[1],
                                  None, "10s", 0.5, 0.5, phase="zero-double", fir_window="hann",
                                  fir_design="firwin2", verbose="error")


outside_breaks = ~spans_to_mask(raw, *break_spans)      # True outside the breaks (sample by sample)


# The standard EEG positions F9 and F10 on MNE's template head, in head coordinates
# (MNE 1.13 renamed the template "standard_1005" to "colin27_1005"; older versions only have the old name)
template = "colin27_1005" if "colin27_1005" in mne.channels.get_builtin_montages() else "standard_1005"
montage = mne.channels.make_standard_montage(template).get_positions()
to_head = mne.transforms.get_ras_to_neuromag_trans(montage["nasion"], montage["lpa"], montage["rpa"])
f9_f10 = mne.transforms.apply_trans(to_head, np.array([montage["ch_pos"]["F9"], montage["ch_pos"]["F10"]]))
# The good Z sensor nearest to each of the two positions
z_pos = head_positions(raw, z_names)
front_pair = [z_names[int(np.argmin(np.linalg.norm(z_pos - p, axis=1)))] for p in f9_f10]

if SURROGATE_CHANNELS == "F9/F10":
    surrogate_choice = "the F9/F10 pair"
    surrogate_pair = front_pair
else:
    surrogate_choice = "set in Step 1"
    surrogate_pair = list(SURROGATE_CHANNELS)
    unknown = [ch for ch in surrogate_pair if ch not in raw.ch_names]
    if unknown or len(surrogate_pair) != 2:
        raise ValueError(f"SURROGATE_CHANNELS must be \"F9/F10\" or two channel names of the recording "
                         f"(unknown: {unknown}).")
print(f"Used for OPM_blink: {' minus '.join(surrogate_pair)} ({surrogate_choice})")

# ---- Map of the sensors, as in preprocessing notebook 01 (Step 5), with the pair marked ------------------------
# All Z channels of the helmet's OPM sensors, bad ones included (drawn in red). Only the header is copied, not the data.
helmet_z = [raw.ch_names[i] for i in mne.pick_types(raw.info, meg="mag", ref_meg=False, exclude=[])
            if raw.ch_names[i].endswith("[Z]")]
info_map = mne.pick_info(raw.info, mne.pick_channels(raw.ch_names, include=helmet_z, ordered=True))
fig, ax = plt.subplots(figsize=(7, 7))
mne.viz.plot_sensors(info_map, kind="topomap", show_names=True, axes=ax, show=False,
                     title="Z sensors seen from above (nose at the top)")
# Where MNE drew each sensor (one dot per channel of info_map, in the same order), to mark the pair on top
dots = ax.collections[0].get_offsets()
if len(dots) == len(info_map.ch_names):
    for ch, marker, which in zip(surrogate_pair, ["^", "v"], ["first", "second"]):
        if ch in info_map.ch_names:
            ax.scatter(*dots[info_map.ch_names.index(ch)], s=300, marker=marker, facecolor="none",
                       edgecolor="tab:green", linewidth=2.2, zorder=5, label=f"{ch} ({which})")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
else:
    print("(The pair could not be marked on the map; it is printed above.)")
plt.show()

### Step 4b: Find the blinks in the surrogate signal

The surrogate signal is the first sensor minus the second. `mne.preprocessing.find_eog_events`
then filters it to `OPM_BLINK_BAND_HZ` (1-10 Hz, its default and FLUX's) and finds its peaks. Two details of how it
works, from MNE's code:

- It returns the time of each blink's **peak**, not the moment the signal crosses the threshold.
- The threshold is how far a peak must rise **above the signal around it**. With `OPM_BLINK_THRESHOLD_FT = None`,
  MNE uses a quarter of the range of the filtered signal: (maximum - minimum) / 4. That adapts to each recording,
  but a single very large artifact can push it too high. FLUX set it by eye, to 3000 fT, for its data. You don't
  need to choose a value: the automatic one is the default. With eye tracking, Step 4c tests a range of thresholds
  against the eye tracker's blinks; in the sample recording the automatic threshold matched almost as well as the
  best of them (F1 0.88 against 0.89).
- It looks for peaks in only **one** direction. The 1-10 Hz filter turns each blink into a down-and-up wave, with one
  swing around the start of the blink and the opposite one as the eyelid closes, and the two can be about equally
  large. MNE picks the direction from the single largest value (positive or negative) in the whole recording, so
  the peaks can fall on either swing. In the sample recording they fall on the first one, at about the eye tracker's
  blink start.

The breaks (`BAD_break`) are left out.

**How long each blink is labelled.** FLUX labels a fixed stretch, 0.25 s before to 0.25 s after each peak. But the
peak is not the middle of the blink: in the sample recording it comes about 0.03 s before the eye tracker's blink
start and 0.64 s before its end, and the blink's field lasts another 0.14 s before and 0.39 s after that (Step 3b).
So ±0.25 s covers less than half of the blinks' field there (39 %, Step 4c). With `OPM_BLINK_WINDOW_S = "auto"` (the
default):

- **with eye tracking**, the window is worked out from Step 3: for each eye-tracker blink with a peak in it (the
  blink plus its pads), the median time from the blink start to its first peak, plus the pad before (Step 3b), gives
  the start; the median time from that peak to the blink end, plus the pad after, gives the end. In the sample
  recording: about -0.10 to +1.07 s, which covers 83 % of the eye tracker's blink time.
- **without eye tracking**, there is nothing to measure it against, so FLUX's ±0.25 s is used. Measuring the window
  from the OPM sensors alone (by averaging around the peaks) does not work well in a task like this one: people blink
  at set moments in the trial (the BLINK screen), so the average around the peaks also contains the task's own
  fields, and they make the window much too long.

**One span per blink.** With the F9/F10 pair, many blinks give two peaks: one at the start and one at the eye
reopening (Step 4c counts them as "extra peaks in a blink"). With a window as long as the measured one, the reopening
peak's own span would reach about a second past the end of the blink. So a peak that falls within the window of the
peak before it is merged into that blink and gets no span of its own. In the sample recording this merges 107 of the
486 peaks, and the share of the `OPM_blink` time that falls outside the eye tracker's blinks drops from 20 % to 12 %;
9 of the 432 blinks lose their label, because they follow the previous blink so closely that their peak is merged.
With FLUX's ±0.25 s, no peaks are that close, so nothing is merged.

Step 4c prints the window measured with eye tracking. For recordings of the same task without eye tracking, you can
set `OPM_BLINK_WINDOW_S` in Step 1 to it, e.g. `(-0.10, 1.07)`.

The plot shows the filtered surrogate signal for 30 s with the peaks found (open triangles: merged into the blink
before them) and, if there are any, the eye tracker's
blinks (darker shading) with the pads from Step 3 before and after each one (lighter shading): together the stretch
Step 4c counts as that blink.

In [ ]:
occ_raw = surrogate_raw(raw, surrogate_pair)
occ_filt = filtered_surrogate(occ_raw)
auto_thresh_T = (occ_filt[outside_breaks].max() - occ_filt[outside_breaks].min()) / 4
opm_thresh_T = auto_thresh_T if OPM_BLINK_THRESHOLD_FT is None else OPM_BLINK_THRESHOLD_FT * 1e-15
opm_peaks = opm_blink_peaks(occ_raw, opm_thresh_T)
print(f"Threshold: {opm_thresh_T * 1e15:.0f} fT ({'MNE automatic' if OPM_BLINK_THRESHOLD_FT is None else 'Step 1'}; "
      f"MNE's automatic value here: {auto_thresh_T * 1e15:.0f} fT)")
print(f"{len(opm_peaks)} peaks found outside breaks")

# ---- How long to label around each peak --------------------------------------------------------------------
# With eye tracking: where the FIRST peak of each eye-tracker blink falls in it (blinks outside breaks), plus the
# Step 3 pads. A peak belongs to a blink if it falls inside ET_blink extended by the pads (where the blink's field
# is there). Later peaks in the same blink (e.g. at the eye reopening) are left out of this measurement.
opm_window_et = None
if HAS_ET_BLINKS:
    ok_s, ok_e = et_blink[0][~et_in_break], et_blink[1][~et_in_break]
    lead, tail = [], []                       # blink start to its first peak, and that peak to the blink end (s)
    for s_k, e_k in zip(ok_s, ok_e):
        inside = opm_peaks[(s_k - blink_pad[0] <= opm_peaks) & (opm_peaks <= e_k + blink_pad[1])]
        if len(inside):
            lead.append(inside.min() - s_k)
            tail.append(e_k - inside.min())
    if len(lead) >= 20:
        opm_window_et = (-round(float(np.median(lead) + blink_pad[0]), 3), round(float(np.median(tail) + blink_pad[1]), 3))
        print(f"{len(lead)} eye-tracker blinks have a peak (with the pads); their first peak comes a median {np.median(lead) * 1000:.0f} ms after the blink "
              f"start and {np.median(tail) * 1000:.0f} ms before its end. With the pads from Step 3 ({blink_pad[0] * 1000:.0f} and "
              f"{blink_pad[1] * 1000:.0f} ms): window {opm_window_et[0]} to {opm_window_et[1]} s around the peak.")
if OPM_BLINK_WINDOW_S != "auto":
    opm_window = tuple(float(x) for x in OPM_BLINK_WINDOW_S)
    print(f"Window around each peak: {opm_window[0]} to {opm_window[1]} s (set in Step 1)")
elif opm_window_et is not None:
    opm_window = opm_window_et
    print(f"Window around each peak: {opm_window[0]} to {opm_window[1]} s (from the eye tracker and Step 3)")
else:
    opm_window = (-0.25, 0.25)
    why = ("fewer than 20 peaks fall in an eye-tracker blink" if HAS_ET_BLINKS else "no eye-tracker blinks to measure it against")
    print(f"Window around each peak: -0.25 to 0.25 s (FLUX's; {why})")

# ---- One span per blink: a peak that falls within the window of the previous kept peak belongs to that blink ------
# (e.g. the second peak at the eye reopening) and gets no span of its own
opm_kept = []
for p in opm_peaks:
    if not opm_kept or p - opm_kept[-1] >= opm_window[1]:
        opm_kept.append(p)
opm_kept = np.array(opm_kept)
print(f"OPM_blink: {len(opm_kept)} blinks ({len(opm_peaks) - len(opm_kept)} peaks merged into the blink before them), "
      f"{len(opm_kept) / (usable_s / 60):.1f} per minute outside breaks")
set_spans(raw, "OPM_blink", opm_kept + opm_window[0], opm_kept + opm_window[1])

# ---- 30 s of the filtered surrogate signal --------------------------------------------------------------
t_show = (opm_peaks[len(opm_peaks) // 2] if len(opm_peaks) else rec_length_s(raw) / 2) - 15
t_show = float(np.clip(t_show, 0, max(rec_length_s(raw) - 30, 0)))
sel = slice(int(t_show * SFREQ), int((t_show + 30) * SFREQ))
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(raw.times[sel], occ_filt[sel] * 1e15, color="k", lw=0.8, label=f"surrogate ({' minus '.join(surrogate_pair)}), "
        f"{OPM_BLINK_BAND_HZ[0]}-{OPM_BLINK_BAND_HZ[1]} Hz")
# Kept peaks (one OPM_blink span each) and merged peaks (inside the window of the peak before them)
merged = np.setdiff1d(opm_peaks, opm_kept)
for times, style, label in [(opm_kept, dict(marker="v", color="tab:orange", ls=""), "OPM_blink peaks"),
                            (merged, dict(marker="v", mfc="none", mec="tab:orange", ls=""), "peaks merged into the blink before")]:
    shown = times[(times >= t_show) & (times < t_show + 30)]
    ax.plot(shown, occ_filt[np.round(shown * SFREQ).astype(int)] * 1e15, label=label, **style)
if HAS_ET_BLINKS:
    # Each eye-tracker blink (darker), with the Step 3 pads before and after it (lighter): the stretch Step 4c
    # counts as that blink
    for s, e in zip(*et_blink):
        if e + blink_pad[1] > t_show and s - blink_pad[0] < t_show + 30:
            ax.axvspan(s - blink_pad[0], s, color=LABEL_COLORS["ET_blink"], alpha=0.08, lw=0)
            ax.axvspan(e, e + blink_pad[1], color=LABEL_COLORS["ET_blink"], alpha=0.08, lw=0)
            ax.axvspan(s, e, color=LABEL_COLORS["ET_blink"], alpha=0.25, lw=0)
    ax.plot([], [], color=LABEL_COLORS["ET_blink"], alpha=0.5, lw=8, label=ET_BLINK_ANNOT)
    ax.plot([], [], color=LABEL_COLORS["ET_blink"], alpha=0.15, lw=8,
            label=f"pads ({blink_pad[0] * 1000:.0f} ms before, {blink_pad[1] * 1000:.0f} ms after; Step 3)")
ax.set(xlim=(t_show, t_show + 30), xlabel="Time (s)", ylabel="fT", title="The surrogate ocular signal and the blinks found")
ax.legend(fontsize=8, loc="upper right")
plt.show()

### Step 4c: How well do the OPM blinks match the eye tracker's?

Only with eye tracking. The eye tracker's blinks outside the breaks are the reference. An `OPM_blink` peak **matches**
a blink when it falls inside that blink's `ET_blink` span extended by the pads from Step 3 (the stretch where Step 3b
found the blink's field, e.g. 142 ms before and 385 ms after in the sample recording); each blink can be matched only
once (see "extra peaks in a blink" below). The pads are what makes a peak just before the eye tracker's blink start, where the peaks of the F9/F10 pair
often fall (Step 4b), still count as that blink. If you set `BLINK_PAD_S` by hand in Step 1, those pads are used
here too; with `(0, 0)` only peaks inside the `ET_blink` span itself count, and most of this pair's peaks would be
counted as false alarms. Then:

| Count | Meaning |
|---|---|
| **hits** | blinks found by both (each blink counts once) |
| **misses** | eye-tracker blinks without an OPM peak |
| **extra peaks in a blink** | further OPM peaks inside a blink that already has one. With the F9/F10 pair these are mostly at the eye reopening: the eyelid's closing and reopening can each give a peak. They are part of a real blink, so they are not false alarms |
| **false alarms** | OPM peaks outside every eye-tracker blink and its pads (these can be real blinks the eye tracker missed, or other eye movements) |
| **sensitivity** | hits / eye-tracker blinks: the share of real blinks found |
| **precision** | (hits + extra peaks in a blink) / OPM peaks: the share of OPM peaks that fall in a real blink |
| **F1** | one number for both: 2 x sensitivity x precision / (sensitivity + precision) |
| **lag** | the median time from the `ET_blink` start to the matching OPM peak |
| **r** | the correlation between the filtered surrogate signal and a signal that is 1 during `ET_blink` and 0 otherwise. It measures how closely the surrogate follows the blinks over time (its sign only depends on which sensor comes first) |

The table shows these numbers for the pair from Step 4a, at the threshold used in Step 4b: what to expect from
recordings without an eye tracker.

A second table shows how much of the eye tracker's blink time (`ET_blink` plus the Step 3 pads) the `OPM_blink`
spans cover, and how much of their time lies outside it, for the window used in Step 4b and for FLUX's ±0.25 s
(each with the peaks merged as in Step 4b).

The plot under the first table repeats the comparison for a range of thresholds for the pair used, and the line under
the plot gives the one that matches best (the highest F1). **Nothing needs to be chosen here**: with eye tracking,
`BAD_blink` comes from the eye tracker anyway. The sweep tells you whether MNE's automatic threshold (the orange
dashed line, used by default) is close to the best one. If it is, as in the sample recording, keep
`OPM_BLINK_THRESHOLD_FT = None` for recordings without eye tracking. If the best value is clearly different and
gives a much higher F1 in several recordings with eye tracking, you can set `OPM_BLINK_THRESHOLD_FT` to it for
similar recordings without.

In [ ]:
def match_blinks(peaks, ref_starts, ref_ends, pad):
    # Match OPM peaks to reference blinks. A peak can belong to a blink if it falls inside the blink's span extended
    # by the pads (pad = (before, after) in s). Each blink is matched by one peak (the nearest unmatched blink the peak
    # falls in); further peaks inside a blink, e.g. at the eye reopening, are counted as "extra peaks in a blink",
    # not as false alarms. The lag is still measured from the blink's own start.
    used = np.zeros(len(ref_starts), bool)
    lags, n_in_blink = [], 0
    for p in np.sort(peaks):
        in_blink = (ref_starts - pad[0] <= p) & (p <= ref_ends + pad[1])
        n_in_blink += bool(in_blink.any())
        inside = np.flatnonzero(in_blink & ~used)
        if len(inside):
            k = inside[np.argmin(np.abs((ref_starts[inside] + ref_ends[inside]) / 2 - p))]
            used[k] = True
            lags.append(p - ref_starts[k])
    hits = int(used.sum())
    sens = hits / len(ref_starts) if len(ref_starts) else np.nan
    prec = n_in_blink / len(peaks) if len(peaks) else np.nan
    f1 = 2 * sens * prec / (sens + prec) if hits else 0.0
    return dict(hits=hits, misses=len(ref_starts) - hits, extra_peaks_in_a_blink=n_in_blink - hits,
                false_alarms=len(peaks) - n_in_blink, sensitivity=round(sens, 3), precision=round(prec, 3),
                F1=round(f1, 3), lag_ms=round(1000 * float(np.median(lags)), 0) if lags else np.nan)


best_thresh_fT = None
if not HAS_ET_BLINKS:
    print("No eye-tracker blinks: nothing to compare with. Step 4c is skipped.")
else:
    ref_s, ref_e = et_blink[0][~et_in_break], et_blink[1][~et_in_break]
    blink_indicator = spans_to_mask(raw, ref_s, ref_e).astype(float)

    # The pair used, at the threshold used in Step 4b
    r = np.corrcoef(occ_filt[outside_breaks], blink_indicator[outside_breaks])[0, 1]
    match_row = {"sensors": " minus ".join(surrogate_pair), "threshold_fT": round(opm_thresh_T * 1e15),
                 "OPM peaks": len(opm_peaks), "ET blinks": len(ref_s),
                 **match_blinks(opm_peaks, ref_s, ref_e, blink_pad), "r": round(r, 2)}
    display(pd.DataFrame([match_row]).set_index("sensors"))

    # ---- Threshold sweep for the pair used ----------------------------------------------------------------
    sweep_fT = np.geomspace(auto_thresh_T / 10, auto_thresh_T * 1.5, 16) * 1e15
    sweep = pd.DataFrame([{"threshold_fT": round(th), **match_blinks(opm_blink_peaks(occ_raw, th * 1e-15), ref_s,
                                                                         ref_e, blink_pad)}
                          for th in sweep_fT])
    best = sweep.loc[sweep["F1"].idxmax()]
    best_thresh_fT = float(best["threshold_fT"])
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(sweep["threshold_fT"], sweep["sensitivity"], "o-", label="sensitivity")
    ax.plot(sweep["threshold_fT"], sweep["precision"], "s-", label="precision")
    ax.plot(sweep["threshold_fT"], sweep["F1"], "k^-", label="F1")
    ax.axvline(opm_thresh_T * 1e15, color="tab:orange", ls="--", label=f"used in 4b ({opm_thresh_T * 1e15:.0f} fT)")
    ax.axvline(best_thresh_fT, color="tab:green", ls=":", label=f"best F1 ({best_thresh_fT:.0f} fT)")
    ax.set(xscale="log", xlabel="Threshold (fT)", ylim=(0, 1.05),
           title=f"Matching the eye tracker: {' minus '.join(surrogate_pair)}")
    ax.legend(fontsize=8)
    plt.show()
    print(f"Best match: {best_thresh_fT:.0f} fT (sensitivity {best['sensitivity']:.2f}, precision {best['precision']:.2f}, "
          f"F1 {best['F1']:.2f}).")

    # ---- How well the OPM_blink spans cover the eye tracker's blinks, for different windows ----------------------
    ref_blink = merge_spans(*subtract_spans(*merge_spans(et_blink[0] - blink_pad[0], et_blink[1] + blink_pad[1]),
                                            *break_spans))
    cover_rows = []
    windows = {"used in 4b": opm_window, "FLUX": (-0.25, 0.25)}
    if opm_window_et is not None:
        windows["from the eye tracker (4b)"] = opm_window_et
    for name, (w0, w1) in windows.items():
        kept_w = []                                # one span per blink, merged as in Step 4b for this window
        for p in opm_peaks:
            if not kept_w or p - kept_w[-1] >= w1:
                kept_w.append(p)
        kept_w = np.array(kept_w)
        opm_sp = subtract_spans(kept_w + w0, kept_w + w1, *break_spans)
        cover_rows.append({"window": name, "start (s)": w0, "end (s)": w1,
                           "eye-tracker blink time covered (%)": round(100 * overlap_s(opm_sp, ref_blink) / total_s(*ref_blink)),
                           "OPM_blink time outside it (%)": round(100 * (1 - overlap_s(opm_sp, ref_blink) / max(total_s(*opm_sp), 1e-9)))})
    print("\nHow much of the eye tracker's blink time (ET_blink plus the Step 3 pads) the OPM_blink spans cover:")
    display(pd.DataFrame(cover_rows).drop_duplicates(subset=["start (s)", "end (s)"]).set_index("window"))
    if opm_window_et is not None:
        print(f"For recordings of this task without eye tracking: OPM_BLINK_WINDOW_S = {opm_window_et} in Step 1.")

### Step 4d: Use the OPM blinks as `BAD_blink` (only without eye tracking, or with `BLINK_SOURCE = "opm"`)

With `BLINK_SOURCE = "opm"`, or without eye tracking, the `OPM_blink` spans become `BAD_blink` (the parts in breaks
cut away, as in Step 3c), and `OPM_blink` is removed so the blinks don't carry two labels.

Otherwise `BAD_blink` comes from the eye tracker (Step 3c). `OPM_blink` then stays in memory for the plots, but it
is **not saved** (Step 10), and because its name doesn't start with `BAD`, it doesn't hide anything from the steps
below.

**The `OCC` channel.** Without eye tracking (or with `ADD_SURROGATE_EOG = True`), Step 10 adds the surrogate signal
to the saved recording as an EOG channel named `OCC`, unfiltered, like FLUX's. A later ICA notebook can use it to find
the blink components (`ica.find_bads_eog(raw, ch_name="OCC")`). MNE's ICA leaves EOG channels out of the
decomposition itself by default, so `OCC` doesn't change the ICA, it is only compared with its components. With eye
tracking it is not added, because the eye tracker's blinks are the better reference (see the introduction).

In [ ]:
if blink_source == "opm":
    s, e = subtract_spans(*get_spans(raw, "OPM_blink"), *bad_spans(raw, exclude=("BAD_blink",)))
    set_spans(raw, "BAD_blink", s, e)
    set_spans(raw, "OPM_blink", [], [])            # removed: the blinks are now BAD_blink
    print(f"BAD_blink from the OPM sensors: {len(s)} spans, {total_s(s, e):.0f} s = "
          f"{100 * total_s(s, e) / usable_s:.1f} % of the time outside breaks")
else:
    print(f"BAD_blink comes from the eye tracker (Step 3c). OPM_blink ({len(get_spans(raw, 'OPM_blink')[0])} spans) "
          "is kept for the plots only and is not saved.")
print("OCC channel added to the saved recording (Step 10):", "yes" if add_occ else "no")

## Step 5: Saccades

A **saccade** is a fast eye movement from one place to another. Saccades affect MEG in two ways:

- The eye is an electrical dipole (the *corneo-retinal dipole*), so when it rotates, its magnetic field changes,
  mostly on the front sensors, with a sign that depends on the direction of the movement.
- The eye muscles fire a short burst at the start of each saccade, the **saccadic spike**. In MEG it depends on the
  size and direction of the saccade and is strongest in the gamma range
  ([Carl et al., 2012, NeuroImage](https://pubmed.ncbi.nlm.nih.gov/21963912/)), so it can look like brain gamma
  activity.

Raw-files notebook 05 put EyeLink's saccades on the MEG timeline as `ET_saccade` (blinks are not included: their saccades are
`ET_blink`).

- **5a** describes them, with their size in degrees of visual angle when the screen size and distance are known
  (from the `StimulusPresentation` part of the `_events.json`). The two-panel plot is in degrees of visual angle,
  with rightward saccades in orange and leftward ones in blue. The left panel shows how far and in which direction
  each saccade moved (end minus start, as if every saccade started at 0). The right panel shows where each one
  landed, relative to the screen centre (0 deg). Together they show, for example, whether the rightward saccades
  land right of the centre or bring the eyes back to it.
- **5b** shows their field: the average around the saccade start, split into leftward and rightward saccades
  (otherwise the opposite fields cancel), and the fast part above 30 Hz, where the spike shows.
- **5c** labels them `BAD_saccade`, **only if** `MARK_SACCADES_BAD = True` in Step 1. It is off by default: in a
  task where people look around, labelling every saccade would remove much of the data, and their effect can often
  be removed with ICA instead. In a task with steady fixation, saccades are departures from fixation and labelling
  the large ones can make sense.

Without `ET_saccade` labels, this step is skipped.

### Step 5a: The eye tracker's saccades

In [ ]:
sacc_size_deg = None
if not HAS_ET_SACCADES:
    print(f"No {ET_SACCADE_ANNOT} labels: Step 5 is skipped.")
else:
    sacc = get_spans(raw, ET_SACCADE_ANNOT)
    sacc_in_break = in_spans(sacc[0], *break_spans)
    sacc_dur_ms = (sacc[1] - sacc[0]) * 1000
    print(f"{len(sacc[0])} {ET_SACCADE_ANNOT} spans, {sacc_in_break.sum()} during breaks; outside breaks "
          f"{(~sacc_in_break).sum() / (usable_s / 60):.1f} per minute")

    # ---- Size and direction from the gaze channels (pixels), at the start and end of each saccade ----------
    sacc_dx = sacc_dy = None
    if HAS_GAZE:
        i0 = np.clip(np.round(sacc[0] * SFREQ).astype(int), 0, raw.n_times - 1)
        i1 = np.clip(np.round(sacc[1] * SFREQ).astype(int), 0, raw.n_times - 1)
        gaze = raw.get_data(picks=["ET_xpos", "ET_ypos"])
        sacc_dx, sacc_dy = gaze[0, i1] - gaze[0, i0], gaze[1, i1] - gaze[1, i0]      # pixels (y: down)
        sacc_xy_px = (gaze[0, i0], gaze[1, i0], gaze[0, i1], gaze[1, i1])            # start x, y and end x, y (pixels)
        del gaze
        # Degrees of visual angle, if the screen's size (m), resolution (pixels) and distance (m) are known
        events_json = bids_path.copy().update(split=None, suffix="events", extension=".json").fpath
        screen = json.loads(events_json.read_text()).get("StimulusPresentation", {}) if events_json.is_file() else {}
        if all(k in screen for k in ("ScreenSize", "ScreenResolution", "ScreenDistance")):
            m_per_px = np.array(screen["ScreenSize"], float) / np.array(screen["ScreenResolution"], float)
            dist_m = np.hypot(sacc_dx * m_per_px[0], sacc_dy * m_per_px[1])
            sacc_size_deg = np.degrees(2 * np.arctan(dist_m / (2 * float(screen["ScreenDistance"]))))
            # Start and end positions in degrees of visual angle from the screen centre (+ = right / up). The gaze
            # channels count pixels from the top-left corner of the screen (ScreenOrigin in the _events.json).
            centre_px = np.array(screen["ScreenResolution"], float) / 2
            view_m = float(screen["ScreenDistance"])
            to_deg_x = lambda px: np.degrees(np.arctan((px - centre_px[0]) * m_per_px[0] / view_m))
            to_deg_y = lambda px: -np.degrees(np.arctan((px - centre_px[1]) * m_per_px[1] / view_m))   # pixels count down
            sacc_xy_deg = (to_deg_x(sacc_xy_px[0]), to_deg_y(sacc_xy_px[1]), to_deg_x(sacc_xy_px[2]), to_deg_y(sacc_xy_px[3]))
            screen_half_deg = (to_deg_x(2 * centre_px[0]), -to_deg_y(2 * centre_px[1]))   # the screen edges
            print(f"Screen: {screen['ScreenSize']} m, {screen['ScreenResolution']} pixels, viewed from "
                  f"{screen['ScreenDistance']} m (from {events_json.name})")
        else:
            print("The screen size and distance are not in the _events.json, so sizes are in pixels only.")

    table = pd.DataFrame({"duration (ms)": sacc_dur_ms})
    if sacc_size_deg is not None:
        table["size (deg)"] = sacc_size_deg
    elif sacc_dx is not None:
        table["size (pixels)"] = np.hypot(sacc_dx, sacc_dy)
    display(table[~sacc_in_break].describe().round(2).T)

    if sacc_dx is not None:
        fig, ax1 = plt.subplots(figsize=(5.5, 3.8))
        size = sacc_size_deg if sacc_size_deg is not None else np.hypot(sacc_dx, sacc_dy)
        ax1.hist(size[~sacc_in_break], bins=50, color="0.5")
        ax1.set(xlabel="Size (deg)" if sacc_size_deg is not None else "Size (pixels)", ylabel="Saccades",
                title="Saccade sizes (outside breaks)")
        fig.tight_layout()
        plt.show()

    # ---- Direction and landing point, in degrees of visual angle, coloured by left / right ---------------------
    if sacc_size_deg is not None:
        x0, y0, x1, y1 = (v[~sacc_in_break] for v in sacc_xy_deg)
        rightward = x1 > x0                          # True: the saccade moved to the right
        groups = [(rightward, "tab:orange", f"rightward (n = {rightward.sum()})"),
                  (~rightward, "tab:blue", f"leftward (n = {(~rightward).sum()})")]
        fig, (ax_m, ax_l) = plt.subplots(1, 2, figsize=(13, 6.2))

        # Left: how far and in which direction each saccade moved (end minus start; 0 = no movement)
        for sel, color, label in groups:
            ax_m.scatter((x1 - x0)[sel], (y1 - y0)[sel], s=8, color=color, alpha=0.5, label=label)
        lim_m = max(1.0, np.nanpercentile(np.abs(np.r_[x1 - x0, y1 - y0]), 99.5) * 1.15)
        ax_m.set(xlim=(-lim_m, lim_m), ylim=(-lim_m, lim_m), aspect="equal",
                 xlabel="horizontal movement (deg, + = right)", ylabel="vertical movement (deg, + = up)",
                 title="Movement of each saccade (end minus start)")

        # Right: where each saccade landed, relative to the screen centre (0 deg = the middle of the screen)
        hw, hh = screen_half_deg
        ax_l.add_patch(plt.Rectangle((-hw, -hh), 2 * hw, 2 * hh, fill=False, color="0.6", lw=1))   # the screen edge
        for sel, color, label in groups:
            ax_l.scatter(x1[sel], y1[sel], s=8, color=color, alpha=0.5, label=label)
        ax_l.plot(0, 0, "k+", ms=14, mew=2, label="screen centre (0 deg)")
        # Centred on the screen centre and zoomed to the landing points (the grey screen edge only shows if they reach it)
        lim_l = max(1.0, np.nanpercentile(np.abs(np.r_[x1, y1]), 99.5) * 1.15)
        ax_l.set(xlim=(-lim_l, lim_l), ylim=(-lim_l, lim_l), aspect="equal",
                 xlabel="horizontal position (deg from the screen centre, + = right)",
                 ylabel="vertical position (deg, + = up)",
                 title=f"Where each saccade landed (screen: ±{hw:.1f} x ±{hh:.1f} deg)")
        for ax in (ax_m, ax_l):
            ax.axhline(0, color="0.8", lw=0.8)
            ax.axvline(0, color="0.8", lw=0.8)
            ax.legend(fontsize=8, loc="upper right")
        fig.tight_layout()
        plt.show()

### Step 5b: The saccades' field

Two averages around the **saccade start** (time 0), from saccades outside breaks and blinks (epochs overlapping a
`BAD_` label are left out):

- **Left:** leftward and rightward saccades separately, smoothed to 40 Hz. The corneo-retinal field changes sign
  with the direction, so the two should look roughly like mirror images on the front sensors.
- **Right:** the fast part (above 30 Hz) of all saccades, as GFP (the strength over all sensors). The saccadic
  spike shows as a short peak right at the saccade start.

Few saccades (a few dozen) give noisy averages. In a task with steady fixation, most saccades are small and their
fields are weak.

In [ ]:
if not (HAS_ET_SACCADES and HAS_GAZE):
    print("Step 5b needs the eye tracker's saccades and gaze channels: skipped.")
else:
    use = ~sacc_in_break
    groups = {"leftward": use & (sacc_dx < 0), "rightward": use & (sacc_dx > 0)}
    fig, (ax_lr, ax_fast) = plt.subplots(1, 2, figsize=(13, 4))
    for (name, sel), color in zip(groups.items(), ["tab:blue", "tab:red"]):
        if sel.sum() < 5:
            print(f"Only {sel.sum()} {name} saccades: not averaged.")
            continue
        ev = locked_average(sacc[0][sel], -0.2, 0.4, (-0.2, -0.05))
        ax_lr.plot(ev.times, ev.copy().pick(BLINK_SENSORS or z_names[:6]).data.T * 1e15, color=color, lw=0.8)
        ax_lr.plot([], [], color=color, label=f"{name} (n = {ev.nave})")
    ax_lr.axvline(0, color="k", lw=0.8)
    ax_lr.set(xlabel="Time from the saccade start (s)", ylabel="Field (fT)",
              title=f"Front sensors ({', '.join((BLINK_SENSORS or z_names[:6])[:3])}, ...), 40 Hz low-pass")
    ax_lr.legend(fontsize=8)

    # The fast part: average without the 40 Hz smoothing, then keep only what is above 30 Hz
    samples = np.unique(np.round(sacc[0][use] * SFREQ).astype(int)) + raw.first_samp
    events = np.column_stack([samples, np.zeros_like(samples), np.ones_like(samples)])
    ev_fast = mne.Epochs(raw, events, tmin=-0.2, tmax=0.3, picks=mag_names, baseline=None, reject_by_annotation=True,
                         preload=False, verbose="error").average()
    ev_fast.filter(30.0, None, method="iir", verbose="error")
    ax_fast.plot(ev_fast.times, gfp(ev_fast) * 1e15, color="k")
    ax_fast.axvline(0, color="tab:green", lw=0.8)
    ax_fast.set(xlabel="Time from the saccade start (s)", ylabel="GFP above 30 Hz (fT)",
                title=f"Fast part, all saccades (n = {ev_fast.nave})", xlim=(-0.1, 0.2))
    fig.tight_layout()
    plt.show()

### Step 5c: Label large saccades `BAD_saccade` (only with `MARK_SACCADES_BAD = True`)

Each `ET_saccade` at least `SACCADE_BAD_MIN_DEG` large (all of them if their size isn't known), plus `SACCADE_PAD_S`
before and after, becomes a `BAD_saccade` span. The parts already labelled (breaks, blinks) are cut away, so each
stretch keeps one label. Step 9c shows how many trials this would remove.

In [ ]:
if not HAS_ET_SACCADES:
    print("No saccades: Step 5c is skipped.")
elif not MARK_SACCADES_BAD:
    set_spans(raw, "BAD_saccade", [], [])
    big = (sacc_size_deg >= SACCADE_BAD_MIN_DEG) if sacc_size_deg is not None else np.ones(len(sacc[0]), bool)
    print(f"MARK_SACCADES_BAD is False: no BAD_saccade labels. ({(big & ~sacc_in_break).sum()} saccades outside breaks "
          f"would be labelled with MARK_SACCADES_BAD = True and SACCADE_BAD_MIN_DEG = {SACCADE_BAD_MIN_DEG}.)")
else:
    big = (sacc_size_deg >= SACCADE_BAD_MIN_DEG) if sacc_size_deg is not None else np.ones(len(sacc[0]), bool)
    s, e = merge_spans(sacc[0][big] - SACCADE_PAD_S[0], sacc[1][big] + SACCADE_PAD_S[1])
    s, e = subtract_spans(s, e, *bad_spans(raw, exclude=("BAD_saccade",)))
    set_spans(raw, "BAD_saccade", s, e)
    print(f"BAD_saccade: {len(s)} spans from {big.sum()} saccades"
          + (f" of at least {SACCADE_BAD_MIN_DEG} deg" if sacc_size_deg is not None else "")
          + f", {total_s(s, e):.0f} s = {100 * total_s(s, e) / usable_s:.1f} % of the time outside breaks")

## Step 6: Muscle activity

Muscles (jaw, neck, face, swallowing) produce brief bursts of fast activity that the OPM sensors pick up, mostly at
high frequencies. FLUX finds them with MNE's `annotate_muscle_zscore`
([FLUX, ArtefactAnnotation](https://github.com/FLUX-pipeline/Cerca/blob/main/ArtefactAnnotation.ipynb); the
frequency band follows [Muthukumaraswamy, 2013](https://doi.org/10.3389/fnhum.2013.00138)). For each sensor:

1. filter to `MUSCLE_BAND_HZ` (110-140 Hz), where there is little brain activity but a lot of muscle activity;
2. take the **envelope** (the size of the fast signal at each moment);
3. turn it into a **z-score** (how many standard deviations above that sensor's average it is).

The z-scores of all sensors are added and divided by the square root of the number of sensors, smoothed with a
4 Hz low-pass filter, and wherever the result is above `MUSCLE_THRESHOLD` (3, as in FLUX) the data are labelled
`BAD_muscle`. Good stretches shorter than `MUSCLE_MIN_LENGTH_GOOD_S` (0.1 s) between two muscle stretches are added
to them.

**How this cell differs from `annotate_muscle_zscore`.** It does exactly those steps, but:

- **one sensor at a time**, so it doesn't need a second copy of the whole recording in memory (MNE's function makes
  one);
- **bad channels are left out** (MNE's function uses all channels of the type; FLUX removes the bad ones first);
- **the power-line harmonics in the band are removed first** (`MUSCLE_NOTCH_HZ`: 120 Hz with 60 Hz power), on this
  working copy only. The saved data are not changed. Steady line noise in the band would raise every sensor's
  average and make the muscle bursts stand out less. Step 6a shows whether there is any.

**One label per stretch.** As in MNE's function, the data already labelled `BAD_...` (breaks, blinks, and saccades
if labelled) are left out of the z-scores, so muscle activity during a blink is part of the blink, not counted
twice.

### Step 6a: Is there line noise in the muscle band?

The spectrum of 120 s from the middle of the recording, averaged over the sensors, from 90 to 160 Hz (the labels are
ignored here: line noise is there all the time, and leaving out every blink would cut the data into pieces too short
for a spectrum), with the muscle
band shaded and the frequencies in `MUSCLE_NOTCH_HZ` marked. A narrow peak at 120 Hz is line noise. If you see one at
another frequency in the band, add it to `MUSCLE_NOTCH_HZ`.

In [ ]:
# 120 s from the middle of the recording (the labels are ignored here: see the text above)
t_mid = rec_length_s(raw) / 2
psd = raw.compute_psd(fmin=90, fmax=160, picks=mag_names, tmin=max(t_mid - 60, 0),
                      tmax=min(t_mid + 60, raw.times[-1]), n_fft=int(4 * SFREQ), reject_by_annotation=False,
                      verbose="error")
power = psd.get_data().mean(axis=0) * 1e30          # T²/Hz -> fT²/Hz
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogy(psd.freqs, power, color="k")
ax.axvspan(*MUSCLE_BAND_HZ, color="tab:purple", alpha=0.15, label=f"muscle band {MUSCLE_BAND_HZ[0]}-{MUSCLE_BAND_HZ[1]} Hz")
for f in MUSCLE_NOTCH_HZ:
    ax.axvline(f, color="tab:red", ls="--", lw=1, label=f"removed for detection: {f} Hz")
ax.set(xlabel="Frequency (Hz)", ylabel="Power (fT²/Hz), mean over sensors",
       title=f"Spectrum {max(t_mid - 60, 0):.0f}-{min(t_mid + 60, raw.times[-1]):.0f} s")
ax.legend(fontsize=8)
plt.show()

# How much the power at each harmonic stands above its neighbourhood (a few Hz on each side)
for f in [f for f in (120, 180) if MUSCLE_BAND_HZ[0] - 10 <= f <= MUSCLE_BAND_HZ[1] + 10]:
    at = power[np.argmin(np.abs(psd.freqs - f))]
    near = power[(np.abs(psd.freqs - f) >= 2) & (np.abs(psd.freqs - f) <= 6)].mean()
    print(f"{f} Hz: {at / near:.1f} x the power 2-6 Hz around it"
          + ("  (line noise)" if at / near > 2 else "  (no clear peak)"))

### Step 6b: Find the muscle bursts

This reads, filters and measures each sensor in turn (a long recording takes a few minutes). The plot shows the
combined score over the whole recording with the threshold, and below it the stretch with the largest burst.

In [ ]:
def muscle_scores(rec, channels):
    # The combined muscle z-score per sample (NaN where data are already labelled BAD), as in annotate_muscle_zscore
    sf = rec.info["sfreq"]
    keep = ~spans_to_mask(rec, *bad_spans(rec))
    n_fft = next_fast_len(rec.n_times)
    total = np.zeros(int(keep.sum()))
    for i, ch in enumerate(channels):
        x = rec.get_data(picks=[ch])[0]
        if MUSCLE_NOTCH_HZ:
            x = mne.filter.notch_filter(x, sf, MUSCLE_NOTCH_HZ, verbose="error")
        x = mne.filter.filter_data(x, sf, MUSCLE_BAND_HZ[0], MUSCLE_BAND_HZ[1], fir_design="firwin",
                                   pad="reflect_limited", verbose="error")
        envelope = np.abs(hilbert(x, N=n_fft)[:rec.n_times])[keep]
        total += (envelope - envelope.mean()) / envelope.std()
        if (i + 1) % 32 == 0 or i + 1 == len(channels):
            print(f"  {i + 1} of {len(channels)} sensors done")
    scores = np.full(rec.n_times, np.nan)
    scores[keep] = mne.filter.filter_data(total / np.sqrt(len(channels)), sf, None, 4.0, verbose="error")
    return scores


print("Measuring the muscle band, one sensor at a time ...")
muscle_score = muscle_scores(raw, mag_names)
is_muscle = np.nan_to_num(muscle_score, nan=-np.inf) > MUSCLE_THRESHOLD
# Good stretches shorter than MUSCLE_MIN_LENGTH_GOOD_S are added to the muscle stretches (as in MNE)
runs, n_runs = label_runs(~is_muscle)
run_length = np.bincount(runs.ravel())
short = np.flatnonzero(run_length < MUSCLE_MIN_LENGTH_GOOD_S * SFREQ)
is_muscle |= np.isin(runs, short[short > 0])
s, e = mask_to_spans(is_muscle, SFREQ)
s, e = subtract_spans(s, e, *bad_spans(raw, exclude=("BAD_muscle",)))     # one label per stretch
set_spans(raw, "BAD_muscle", s, e)
print(f"BAD_muscle: {len(s)} spans, {total_s(s, e):.1f} s = {100 * total_s(s, e) / usable_s:.1f} % of the time outside breaks")

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(raw.times, muscle_score, color="k", lw=0.5)
ax.axhline(MUSCLE_THRESHOLD, color="tab:purple", ls="--", label=f"threshold {MUSCLE_THRESHOLD}")
for si, ei in zip(s, e):
    ax.axvspan(si, ei, color=LABEL_COLORS["BAD_muscle"], alpha=0.3, lw=0)
ax.set(xlabel="Time (s)", ylabel="muscle z-score", xlim=(0, rec_length_s(raw)),
       title="Muscle score (gaps: data already labelled BAD); shaded: BAD_muscle")
ax.legend(fontsize=8, loc="upper right")
plt.show()

if len(s):
    k = int(np.nanargmax(np.where(is_muscle, muscle_score, np.nan)))
    centre = k / SFREQ
    plot_window(raw, centre - 2, centre + 2, z_names[::max(len(z_names) // 8, 1)][:8],
                ["BAD_muscle", "BAD_blink", "BAD_break"], f"The largest muscle burst, at {centre:.1f} s", gaze=False)

### Step 6c: Overlap with the eye labels

By design, `BAD_muscle` doesn't overlap `BAD_blink` (or `BAD_saccade`). But saccades that are **not** labelled bad
(`MARK_SACCADES_BAD = False`) are not left out, and their spike is fast activity too, so some may have been labelled
muscle. This counts how many saccade starts fall inside `BAD_muscle`. If it is a large share, the muscle detector is
partly finding saccades.

In [ ]:
muscle = get_spans(raw, "BAD_muscle")
rows = [{"label": name, "overlap with BAD_muscle (s)": round(overlap_s(get_spans(raw, name), muscle), 3)}
        for name in ("BAD_blink", "BAD_saccade", "BAD_break")]
display(pd.DataFrame(rows))
if HAS_ET_SACCADES:
    in_muscle = in_spans(sacc[0], *muscle)
    print(f"{in_muscle.sum()} of {len(sacc[0])} saccade starts ({100 * in_muscle.mean():.1f} %) fall inside BAD_muscle; "
          f"{len(muscle[0])} BAD_muscle spans contain a saccade start: "
          f"{sum(np.any((sacc[0] >= si) & (sacc[0] < ei)) for si, ei in zip(*muscle))}")

## Step 7: Jumps and other bad segments

The last checks catch what the earlier steps didn't explain: head movements, sensor glitches, sudden steps in a
single sensor, and so on. Both follow Harrison Ritz's mne-opm pipeline
([bad_segments.py](https://github.com/harrisonritz/mne-opm/blob/dev/src/custom/preprocessing/bad_segments.py)),
which runs them twice: once before the noise reduction (lenient: 1 s windows, 20 % of the sensors) and once after
it (strict: 0.5 s windows, 5 %). This notebook comes after preprocessing notebook 03's noise reduction, so it uses the strict
settings. Preprocessing notebook 01's checks already cover the coarse problems the lenient pass is for (flat or noisy channels,
breaks).

As before, the stretches already labelled are left out, so these labels only mark what is left.

### Step 7a: Sudden jumps in single sensors (`BAD_jump`, off by default)

OPM sensors can jump suddenly, for example when the field changes too fast for a sensor to follow. MNE's
`annotate_amplitude` finds these from the change between **two consecutive samples**: wherever a sensor changes by
more than `JUMP_THRESHOLD_PT` from one sample to the next, for at least 5 ms in a row (MNE's default `min_duration`),
the data are labelled (MNE calls the label `BAD_peak`; here it is renamed `BAD_jump`, which says what it is). This
runs on the data as they are, not on a filtered copy: filtering would smooth the jumps out.

**`bad_percent=100`.** If a sensor's jumps cover more than `bad_percent` of the recording (MNE's default: 5 %),
`annotate_amplitude` marks the **sensor** bad instead of labelling the times. Here `bad_percent=100` turns that off,
as in mne-opm, because choosing bad sensors is preprocessing notebook 01's job: this notebook never changes the bad-channel list.
Instead, the cell lists any sensor whose jumps cover more than 5 % of the recording. Such a sensor is better marked
bad in preprocessing notebook 01 than having its jumps remove time from every sensor.

**Choosing the threshold.** It depends on the sensors and the data, so it is off (`None`) until you choose one. The
table below always shows the largest sample-to-sample change of each sensor (outside the data already labelled), to
help: a few sensors far above the rest have jumps. mne-opm's example uses 100 pT.

In [ ]:
def largest_steps(rec, channels):
    # Each sensor's largest change between two consecutive samples (pT), outside the data already labelled BAD
    keep = ~spans_to_mask(rec, *bad_spans(rec))
    both = keep[1:] & keep[:-1]
    rows = []
    for ch in channels:
        step = np.abs(np.diff(rec.get_data(picks=[ch])[0]))[both]
        rows.append({"channel": ch, "largest step (pT)": step.max() * 1e12, "99.99th percentile (pT)":
                     np.percentile(step, 99.99) * 1e12})
    return pd.DataFrame(rows).sort_values("largest step (pT)", ascending=False).reset_index(drop=True)


def find_jumps(rec, channels):
    # BAD_jump spans from annotate_amplitude (one sensor at a time, bad_percent=100), and each sensor's share
    starts, ends, share = [], [], {}
    for ch in channels:
        annot, _ = mne.preprocessing.annotate_amplitude(rec, peak=JUMP_THRESHOLD_PT * 1e-12, bad_percent=100,
                                                         picks=[ch], verbose="error")
        s = annot.onset - span_shift(rec)
        starts.extend(s)
        ends.extend(s + annot.duration)
        share[ch] = 100 * annot.duration.sum() / rec_length_s(rec)
    s, e = merge_spans(starts, ends)
    s, e = subtract_spans(s, e, *bad_spans(rec, exclude=("BAD_jump",)))
    return s, e, pd.Series(share, name="% of the recording")


steps = largest_steps(raw, mag_names)
print(f"Largest sample-to-sample change per sensor (the 10 largest; median over all sensors: "
      f"{steps['largest step (pT)'].median():.1f} pT):")
display(steps.head(10).round(2))

if JUMP_THRESHOLD_PT is None:
    set_spans(raw, "BAD_jump", [], [])
    print("JUMP_THRESHOLD_PT is None (Step 1): no BAD_jump labels.")
else:
    s, e, jump_share = find_jumps(raw, mag_names)
    set_spans(raw, "BAD_jump", s, e)
    print(f"BAD_jump ({JUMP_THRESHOLD_PT} pT): {len(s)} spans, {total_s(s, e):.2f} s")
    over = jump_share[jump_share > 5]
    if len(over):
        print(f"These sensors' jumps cover more than 5 % of the recording; consider marking them bad in preprocessing notebook 01:")
        display(over.round(1).to_frame())

### Step 7b: Other bad segments (`BAD_segment`)

The test that mne-opm uses, from the osl-ephys package (`bad_segments` with `metric="std"` and
`channel_wise=True`):

1. A **copy** of the data is filtered to `SEGMENT_BAND_HZ` (1-40 Hz), the band you will usually analyse, so that
   slow drifts and fast noise don't decide what counts as unusual (mne-opm uses its analysis band too). Each stretch
   between labelled data is filtered separately (mne-opm: `skip_by_annotation`), so the large signals in the
   labelled data don't spill into their neighbours.
2. The data that are left (outside the labels) are put end to end and cut into windows of `SEGMENT_WINDOW_S`
   (0.5 s), and the **standard deviation** of each sensor in each window is computed. A window can contain the end
   of one stretch and the start of the next (e.g. on either side of a blink), and the two can be at different
   levels; the jump between them would make the window look unusual. So within a window, each stretch's own mean is
   removed before the spread is measured. (osl-ephys doesn't do this, but in mne-opm blinks are not `BAD`, so its
   windows rarely cross a gap. A window within one stretch gets exactly the osl-ephys value.)
3. For each sensor, the **GESD test** (*generalized extreme studentized deviate*, Rosner, 1983) finds the windows
   whose standard deviation is an outlier. It takes the value farthest from the mean, measures how far it is in
   standard deviations, removes it, and repeats (up to 10 % of the windows). Then it compares each of those
   distances with the value expected by chance at the `SEGMENT_SIGNIFICANCE` level (0.05): all values up to the last
   one above it are outliers. Unlike a fixed z-score cut-off, a few large outliers don't hide smaller ones by
   inflating the standard deviation.
4. A window is **bad** when at least `SEGMENT_CHANNEL_THRESHOLD` (5 %) of the sensors are outliers in it.

The bad windows become `BAD_segment`. The plot shows, for each window, the share of sensors that are outliers, with
the threshold, and below it the stretch with the most sensors affected.

The cell needs a few minutes for a long recording: it reads the data a stretch at a time.

In [ ]:
def find_bad_segments(rec, channels, channel_threshold):
    # mne-opm / osl-ephys bad-segment test (see the text above). Returns the BAD_segment spans and, for the plot,
    # the time of each window and the share of sensors that are outliers in it.
    sf = rec.info["sfreq"]
    keep = ~spans_to_mask(rec, *bad_spans(rec))
    kept_idx = np.flatnonzero(keep)
    win = int(round(SEGMENT_WINDOW_S * sf))
    n_keep = len(kept_idx)
    n_win = int(np.ceil(n_keep / win))
    sum_sq_dev = np.zeros((len(channels), n_win))        # summed squared deviations from each piece's own mean
    counts = np.bincount(np.arange(n_keep) // win, minlength=n_win)
    run_starts, run_ends = mask_to_spans(keep, 1.0)                # sample numbers of the stretches left
    position = 0                                                    # samples left before this stretch
    for r0, r1 in zip(run_starts.astype(int), run_ends.astype(int)):
        n = r1 - r0
        window_of = (position + np.arange(n)) // win
        piece_starts = np.flatnonzero(np.r_[True, np.diff(window_of) != 0])   # where each window's piece starts
        piece_n = np.diff(np.r_[piece_starts, n])
        group = max(1, int(5e7 // n))                                  # sensors read together (limits memory)
        for g0 in range(0, len(channels), group):
            x = rec.get_data(picks=channels[g0:g0 + group], start=r0, stop=r1)
            x = mne.filter.filter_data(x, sf, SEGMENT_BAND_HZ[0], SEGMENT_BAND_HZ[1], method="iir", verbose="error")
            piece_sum = np.add.reduceat(x, piece_starts, axis=1)
            piece_sum_sq = np.add.reduceat(x ** 2, piece_starts, axis=1)
            sum_sq_dev[g0:g0 + group, window_of[piece_starts]] += piece_sum_sq - piece_sum ** 2 / piece_n
        position += n
    std = np.sqrt(np.clip(sum_sq_dev / counts, 0, None))
    bad_win, outliers = bad_windows_channel_wise(std, channel_threshold, {"alpha": SEGMENT_SIGNIFICANCE})
    # Bad windows -> spans of time (a window that bridges a labelled stretch covers it; that part is cut away below)
    bad_kept = bad_win[np.arange(n_keep) // win]
    b0, b1 = mask_to_spans(bad_kept, 1.0)
    starts = kept_idx[b0.astype(int)] / sf
    ends = (kept_idx[b1.astype(int) - 1] + 1) / sf
    starts, ends = subtract_spans(starts, ends, *bad_spans(rec, exclude=("BAD_segment",)))
    window_times = kept_idx[np.minimum(np.arange(n_win) * win + win // 2, n_keep - 1)] / sf
    return starts, ends, window_times, outliers.mean(axis=0)


def plot_segments(rec, window_times, share, threshold, spans, title):
    fig, ax = plt.subplots(figsize=(13, 3.2))
    # Leave a gap in the line where data were left out (e.g. a break), instead of joining across it
    gap = np.r_[np.diff(window_times) > 1.5 * SEGMENT_WINDOW_S, False]
    x = np.insert(window_times.astype(float), np.flatnonzero(gap) + 1, np.nan)
    y = np.insert(100 * share.astype(float), np.flatnonzero(gap) + 1, np.nan)
    ax.plot(x, y, color="k", lw=0.6)
    ax.axhline(100 * threshold, color="tab:pink", ls="--", label=f"threshold {100 * threshold:g} % of the sensors")
    for si, ei in zip(*spans):
        ax.axvspan(si, ei, color=LABEL_COLORS["BAD_segment"], alpha=0.4, lw=0)
    ax.set(xlabel="Time (s)", ylabel="% of sensors\nunusual", xlim=(0, rec_length_s(rec)), title=title)
    ax.legend(fontsize=8, loc="upper right")
    plt.show()


print("Finding bad segments ...")
s, e, seg_times, seg_share = find_bad_segments(raw, mag_names, SEGMENT_CHANNEL_THRESHOLD)
set_spans(raw, "BAD_segment", s, e)
print(f"BAD_segment: {len(s)} spans, {total_s(s, e):.1f} s = {100 * total_s(s, e) / usable_s:.1f} % of the time outside breaks")
plot_segments(raw, seg_times, seg_share, SEGMENT_CHANNEL_THRESHOLD, (s, e),
              f"Share of sensors with an unusual {SEGMENT_WINDOW_S:g} s window ({SEGMENT_BAND_HZ[0]}-{SEGMENT_BAND_HZ[1]} Hz copy); "
              "shaded: BAD_segment")
if len(s):
    centre = seg_times[int(np.argmax(seg_share))]
    plot_window(raw, centre - 2, centre + 2, z_names[::max(len(z_names) // 8, 1)][:8],
                ["BAD_segment", "BAD_muscle", "BAD_blink", "BAD_break"],
                f"The window with the most unusual sensors, at {centre:.1f} s")

## Step 8: The empty-room recording

The **empty-room recording** (`task-noise`) was made with nobody in the helmet. Later steps use it to describe the
sensor and environment noise, most importantly for the **noise covariance** in source modelling: it tells the source
model how large and how correlated the noise on the sensors is. A short glitch in the empty room (a door, a passing
car, a sensor jump) would make that noise look larger than it is for the whole session, so it is worth leaving out.

This step runs the same checks as Step 7 on it: `BAD_jump` (if `JUMP_THRESHOLD_PT` is set) and `BAD_segment`. It
uses a more lenient share of sensors, `EMPTY_ROOM_CHANNEL_THRESHOLD` (30 %, mne-opm's value for the empty room). The
empty room has no blinks, saccades or muscles, so Steps 3-6 don't apply to it.

It reads the empty room that preprocessing notebook 03's `_meg.json` links (`AssociatedEmptyRoom`), with its own bad channels. If
there is none, or `PROCESS_EMPTY_ROOM = False` in Step 1, this step is skipped.

In [ ]:
er = None
if not PROCESS_EMPTY_ROOM:
    print("PROCESS_EMPTY_ROOM is False in Step 1: Step 8 is skipped.")
else:
    try:
        er_path = bids_path.copy().update(split=None).find_empty_room(use_sidecar_only=True)
    except Exception as err:
        er_path = None
        print("No empty-room recording found from the _meg.json (AssociatedEmptyRoom):", err)
    if er_path is None:
        print("Step 8 is skipped.")
    else:
        er = mne_bids.read_raw_bids(er_path)   # "Did not find any events.tsv" and "participants.tsv": expected
        er.load_data()
        er_base = er_path.copy().update(split=None, suffix=None, extension=None).basename
        er_mag_names = [er.ch_names[i] for i in mne.pick_types(er.info, meg="mag", ref_meg=False, exclude="bads")]
        print(f"Loaded: {er_path.fpath.name} ({er.times[-1]:.0f} s, {len(er_mag_names)} good OPM channels, "
              f"bad: {er.info['bads'] or 'none'})")

        if JUMP_THRESHOLD_PT is not None:
            s, e, er_jump_share = find_jumps(er, er_mag_names)
            set_spans(er, "BAD_jump", s, e)
            print(f"BAD_jump: {len(s)} spans, {total_s(s, e):.2f} s")
        s, e, er_seg_times, er_seg_share = find_bad_segments(er, er_mag_names, EMPTY_ROOM_CHANNEL_THRESHOLD)
        set_spans(er, "BAD_segment", s, e)
        print(f"BAD_segment: {len(s)} spans, {total_s(s, e):.1f} s = {100 * total_s(s, e) / rec_length_s(er):.1f} % of the empty room")
        plot_segments(er, er_seg_times, er_seg_share, EMPTY_ROOM_CHANNEL_THRESHOLD, (s, e),
                      "Empty room: share of sensors with an unusual window; shaded: BAD_segment")

## Step 9: Review

### Step 9a: How much data each label covers

For each label: how many spans and how much time, as a share of the whole recording and of the time outside the
breaks. `OPM_blink` (only when blinks come from the eye tracker) is listed for comparison: it is not saved and not
`BAD`. The last row is everything that MNE would leave out (all `BAD...` labels together).

In [ ]:
REVIEW_LABELS = ["BAD_break", "BAD_blink", "BAD_saccade", "BAD_muscle", "BAD_jump", "BAD_segment"]
rows = []
for name in REVIEW_LABELS + (["OPM_blink"] if len(get_spans(raw, "OPM_blink")[0]) else []):
    s, e = get_spans(raw, name)
    rows.append({"label": name, "spans": len(s), "seconds": round(total_s(s, e), 1),
                 "% of recording": round(100 * total_s(s, e) / rec_length_s(raw), 2),
                 "% outside breaks": round(100 * total_s(s, e) / usable_s, 2) if name != "BAD_break" else np.nan})
all_bad = bad_spans(raw)
rows.append({"label": "all BAD together", "spans": len(all_bad[0]), "seconds": round(total_s(*all_bad), 1),
             "% of recording": round(100 * total_s(*all_bad) / rec_length_s(raw), 2),
             "% outside breaks": round(100 * (total_s(*all_bad) - total_s(*break_spans)) / usable_s, 2)})
display(pd.DataFrame(rows).set_index("label"))

### Step 9b: No stretch has two labels

The overlap, in seconds, between every pair of `BAD_` labels. All should be 0 (each step left out what earlier
steps had labelled).

In [ ]:
spans_by_label = {name: get_spans(raw, name) for name in REVIEW_LABELS}
overlaps = pd.DataFrame(0.0, index=REVIEW_LABELS, columns=REVIEW_LABELS)
for i, a in enumerate(REVIEW_LABELS):
    for b in REVIEW_LABELS[i + 1:]:
        overlaps.loc[a, b] = overlaps.loc[b, a] = round(overlap_s(spans_by_label[a], spans_by_label[b]), 3)
display(overlaps)
print("No stretch has two BAD labels ✓" if (overlaps.values == 0).all() else "Some labels overlap ✗ (see the table)")

### Step 9c: How many trials would be lost

A preview of what the labels will cost once the data are cut into epochs. For each event in `PREVIEW_EPOCHS`
(Step 1), the start and end given there make the **provisional epoch window**: the epoch you plan to use. Each event
can have its own window. In the sample task a trial is the deviant with the standard tone just before it (1.11-1.32 s
earlier), so the window is -1.52 to 0.8 s around the deviant: from 0.2 s before that standard to 0.8 s after the
deviant. A long window overlaps more labels, so it loses more trials than a short one would. The real window is set later, when you epoch the data. MNE drops an epoch when any part of it
overlaps a `BAD` label, so the table counts, for each label, the epochs that overlap it, and in the last row the
epochs any label would drop. An epoch can overlap several labels, so the rows don't add up to the last one.

**Without the blinks.** Most analyses won't drop the trials with blinks: they remove the blinks' field with ICA
instead (a later notebook; see "Blinks and ICA" in the introduction) and ignore `BAD_blink` when epoching. The last
two rows show that case: the epochs that the other labels (breaks, saccades, muscle, jumps, bad segments) would
drop, and how many would be left.

In [ ]:
event_names = [n for n in PREVIEW_EPOCHS if n in labels_present]
missing = [n for n in PREVIEW_EPOCHS if n not in labels_present]
if missing:
    print(f"Not in the recording: {missing}. Events it has:",
          sorted(n for n in labels_present if not n.lower().startswith(("bad", "et_", "edge"))))
trial_loss = {}
if not event_names:
    print("None of the events in PREVIEW_EPOCHS is in the recording: Step 9c is skipped.")
else:
    table = {}
    trial_loss = {}           # for the summary: event -> (epochs, dropped by any label, dropped without BAD_blink)
    for ev_name in event_names:
        t_ev = get_spans(raw, ev_name)[0]
        tmin, tmax = PREVIEW_EPOCHS[ev_name]
        win_s, win_e = t_ev + tmin, t_ev + tmax
        col = {}
        any_hit = np.zeros(len(t_ev), bool)
        no_blink_hit = np.zeros(len(t_ev), bool)
        for name in REVIEW_LABELS:
            ls_, le_ = merge_spans(*spans_by_label[name])
            hit = np.array([np.any((ls_ < we) & (le_ > ws)) for ws, we in zip(win_s, win_e)], bool)
            any_hit |= hit
            if name != "BAD_blink":
                no_blink_hit |= hit
            col[name] = f"{hit.sum()} ({100 * hit.mean():.0f} %)"
        col["any BAD label"] = f"{any_hit.sum()} ({100 * any_hit.mean():.0f} %)"
        col["epochs left"] = f"{(~any_hit).sum()} of {len(t_ev)}"
        col["any BAD label except BAD_blink"] = f"{no_blink_hit.sum()} ({100 * no_blink_hit.mean():.0f} %)"
        col["epochs left, blinks kept"] = f"{(~no_blink_hit).sum()} of {len(t_ev)}"
        table[f"{ev_name} ({tmin:g} to {tmax:g} s)"] = col
        trial_loss[ev_name] = (len(t_ev), int(any_hit.sum()), int(no_blink_hit.sum()))
    print("Epochs that would be dropped (each event with its window from PREVIEW_EPOCHS):")
    display(pd.DataFrame(table))

### Step 9d: Overview of the whole recording

One row per label, across the whole recording, with the task events as ticks at the top. Look for labels that
cluster (e.g. muscle or bad segments towards the end of a block) or that cover a lot of one part of the recording.

In [ ]:
show = ["BAD_break", ET_BLINK_ANNOT, "BAD_blink", "OPM_blink", ET_SACCADE_ANNOT, "BAD_saccade", "BAD_muscle",
        "BAD_jump", "BAD_segment"]
show = [n for n in show if len(get_spans(raw, n)[0])]
task_events = sorted(n for n in labels_present if not n.lower().startswith(("bad", "et_", "edge")))
fig, ax = plt.subplots(figsize=(13, 0.45 * (len(show) + 1) + 1.2))
for row, name in enumerate(show):
    s, e = get_spans(raw, name)
    ax.broken_barh(list(zip(s, np.maximum(e - s, 0.05))), (row - 0.4, 0.8), color=LABEL_COLORS.get(name, "0.5"))
if task_events:
    t_task = np.concatenate([get_spans(raw, n)[0] for n in task_events])
    ax.eventplot(t_task, lineoffsets=len(show), linelengths=0.8, color="k", linewidths=0.3)
ax.set_yticks(range(len(show) + (1 if task_events else 0)), show + (["task events"] if task_events else []))
ax.set(xlim=(0, rec_length_s(raw)), xlabel="Time (s)", title=f"{base}: all labels")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

### Step 9e: Browse the data (optional)

With `SHOW_BROWSER = True` in Step 1, this opens MNE's interactive browser, with all labels shaded and the
surrogate channel's sensors and the blink sensors first. On scotty every redraw goes over the tunnel, so keep
`BROWSE_DURATION_S` small if it feels slow.

In [ ]:
if SHOW_BROWSER:
    order = [raw.ch_names.index(ch) for ch in dict.fromkeys(list(surrogate_pair) + BLINK_SENSORS + mag_names)]
    order += [raw.ch_names.index(ch) for ch in ("ET_xpos", "ET_ypos", "ET_pupil") if ch in raw.ch_names]
    raw.plot(order=order, duration=BROWSE_DURATION_S, n_channels=20, scalings=dict(mag=4e-12, eyegaze=2e3, pupil=2e3))
else:
    print("SHOW_BROWSER is False (Step 1): no browser.")

## Step 10: Save the labelled recording

The labels so far exist only in memory. This step saves a copy of the recording **with the labels** in
`derivatives/annotated/` (the folder name is `ANNOT_DERIV` in Step 1), labelled `proc-annot`, like preprocessing
notebooks 02 (`proc-filt`) and 03 (`proc-denoise`):

```
derivatives/annotated/sub-01/ses-01/meg/sub-01_ses-01_task-oddballftcued_proc-annot_meg.fif
```

**The data are not changed**: they are preprocessing notebook 03's, sample for sample. What's new is in the sidecar
files, and the next notebook gets all of it by loading this recording with `read_raw_bids`:

| File | What's in it |
|---|---|
| `_events.tsv` | preprocessing notebook 03's events (task events, eye-tracking labels, `BAD_break`) **plus this notebook's labels**: `BAD_blink`, `BAD_saccade`, `BAD_muscle`, `BAD_jump`, `BAD_segment`. `read_raw_bids` builds the annotations from this file only |
| `_events.json` | preprocessing notebook 03's, plus what each new label means and, under `ArtifactDetection`, every setting this notebook used |
| `_channels.tsv` | preprocessing notebook 03's, plus the `OCC` channel if it was added |
| `_meg.json` | preprocessing notebook 03's; **`Sources`** lists the files this recording was made from; **`AssociatedEmptyRoom`** points to the labelled empty room (Step 10b) |
| `_channels.json`, `_coordsystem.json` | copied from preprocessing notebook 03 |
| `dataset_description.json` | in `derivatives/annotated/`: what made this folder, and links to the `raw` and `denoise` datasets |

**`OCC`** (only when `add_occ` is on, see Step 2): the surrogate ocular signal from Step 4 (first sensor minus
second, unfiltered), added as an EOG channel.

`OPM_blink` is not saved when the blinks come from the eye tracker: it was only used for the comparison in Step 4c.

The copy takes as much disk space as preprocessing notebook 03's recording; a large one is split into
`..._split-01_meg.fif`, `..._split-02_meg.fif`, as before. Saving it to the server can take a few minutes. If it
exists from an earlier run, this step stops, unless `OVERWRITE = True` in Step 1.

In [ ]:
import shutil

OUTPUT_PROC = "annot"
SAVED_LABELS = ["BAD_blink", "BAD_saccade", "BAD_muscle", "BAD_jump", "BAD_segment"]
annot_root = BIDS_ROOT / "derivatives" / ANNOT_DERIV
out_dir = annot_root / sub_dir / "meg"
out_base = f"{base}_proc-{OUTPUT_PROC}"          # every file name of the new recording starts with this
out_fif = out_dir / f"{out_base}_meg.fif"
if sorted(out_dir.glob(f"{out_base}*_meg.fif")) and not OVERWRITE:
    raise FileExistsError(f"{out_base}_meg.fif exists from an earlier run. Set OVERWRITE = True in Step 1 to replace it.")
out_dir.mkdir(parents=True, exist_ok=True)


def uri(dataset, file):
    # A BIDS URI, e.g. "bids:denoise:sub-01/ses-01/meg/..._meg.fif" ("denoise" as named in DatasetLinks)
    return f"bids:{dataset}:{(sub_dir / 'meg' / Path(file).name).as_posix()}"


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, content):
    Path(path).write_text(json.dumps(content, indent=4, ensure_ascii=False), encoding="utf-8")


def label_rows(rec, labels, columns):
    # One _events.tsv row per span of the given labels (onset from the first sample), other columns "n/a"
    spans = [(s, e - s, name) for name in labels for s, e in zip(*get_spans(rec, name))]
    rows = pd.DataFrame({c: "n/a" for c in columns}, index=range(len(spans)))
    rows["onset"] = [f"{s:.6f}" for s, _, _ in spans]
    rows["duration"] = [f"{d:.6f}" for _, d, _ in spans]
    rows["trial_type"] = [name for _, _, name in spans]
    return rows


def sorted_events(table, sfreq):
    # Sorted by onset, with the sample column (if any) recomputed at this sampling rate
    if "sample" in table.columns:
        table["sample"] = [str(int(round(float(x) * sfreq))) for x in table["onset"]]
    return table.iloc[np.argsort(table["onset"].astype(float).to_numpy(), kind="stable")]


LEVELS = {
    "BAD_blink": "An eye blink: the eye tracker's blink (ET_blink) plus padding, or a blink found from the OPM sensors "
                 "(surrogate ocular channel); see ArtifactDetection",
    "BAD_saccade": "A saccade (eye movement) from the eye tracker (ET_saccade) at least SaccadeBadMinDeg large, plus padding",
    "BAD_muscle": "Muscle activity: high z-scored envelope in the muscle band on the OPM sensors",
    "BAD_jump": "A sudden jump in a sensor: sample-to-sample change above the threshold (mne.preprocessing.annotate_amplitude)",
    "BAD_segment": "Unusual data: windows where enough sensors are GESD outliers in standard deviation (osl-ephys, as in mne-opm)",
}
task_settings = {
    "Description": "Artifact labels added by preprocessing notebook 04 (04_Artifact_Detection.ipynb, Pygers-MEG). "
                   "Each stretch of data has at most one BAD label.",
    "Order": "breaks (input), blinks, saccades, muscle, jumps, segments; each step skips data already labelled BAD",
    "Channels": f"{len(mag_names)} good OPM magnetometer channels (the input's bad channels left out)",
    "BlinkSource": "eye tracker (ET_blink)" if blink_source == "eyetracker" else "OPM sensors (surrogate ocular channel)",
    "BlinkPadSeconds": list(blink_pad) if blink_source == "eyetracker" else None,
    "BlinkPadFrom": ("measured (Step 3b)" if BLINK_PAD_S == "auto" else "set by hand") if blink_source == "eyetracker" else None,
    "SurrogateChannels": list(surrogate_pair),
    "SurrogateChannelsChoice": surrogate_choice,
    "SurrogateChannelAdded": bool(add_occ),
    "OPMBlinkThresholdfT": round(opm_thresh_T * 1e15, 1),
    "OPMBlinkBandHz": list(OPM_BLINK_BAND_HZ), "OPMBlinkWindowSeconds": list(opm_window), "OPMBlinkWindowFrom": ("set by hand" if OPM_BLINK_WINDOW_S != "auto" else "eye tracker and Step 3 pads"
                           if opm_window_et is not None else "FLUX default"),
    "OPMBlinkBestMatchThresholdfT": best_thresh_fT,
    "MarkSaccadesBad": MARK_SACCADES_BAD, "SaccadeBadMinDeg": SACCADE_BAD_MIN_DEG, "SaccadePadSeconds": list(SACCADE_PAD_S),
    "MuscleBandHz": list(MUSCLE_BAND_HZ), "MuscleNotchHz": list(MUSCLE_NOTCH_HZ), "MuscleThresholdZ": MUSCLE_THRESHOLD,
    "MuscleMinLengthGoodSeconds": MUSCLE_MIN_LENGTH_GOOD_S,
    "JumpThresholdpT": JUMP_THRESHOLD_PT,
    "SegmentBandHz": list(SEGMENT_BAND_HZ), "SegmentWindowSeconds": SEGMENT_WINDOW_S,
    "SegmentChannelThreshold": SEGMENT_CHANNEL_THRESHOLD, "SegmentSignificance": SEGMENT_SIGNIFICANCE,
    "MNEVersion": mne.__version__,
}

# ---- dataset_description.json -------------------------------------------------------------------------
raw_version = read_json(BIDS_ROOT / "dataset_description.json").get("BIDSVersion")
write_json(annot_root / "dataset_description.json", {
    "Name": "OPM-MEG with artifact labels",
    "BIDSVersion": raw_version,
    "DatasetType": "derivative",
    "GeneratedBy": [{"Name": "04_Artifact_Detection.ipynb (Pygers-MEG)",
                     "Description": "Adds artifact labels (BAD_ annotations, in each recording's _events.tsv) to "
                                    "preprocessing notebook 03's recordings; the data are not changed"}],
    "DatasetLinks": {"raw": "../../", DENOISE_DERIV: f"../{DENOISE_DERIV}/"},
})

# ---- The FIF file(s): the recording with its labels (and OCC) ---------------------------------------------
denoise_fifs = [Path(f) for f in raw.filenames]          # preprocessing notebook 03's file(s), read in Step 2
opm_blink_spans = get_spans(raw, "OPM_blink")
set_spans(raw, "OPM_blink", [], [])                     # not saved (put back below)
if "OCC" in raw.ch_names:                               # from an earlier run of this cell
    raw.drop_channels(["OCC"])
if add_occ:
    raw.add_channels([surrogate_raw(raw, surrogate_pair).set_annotations(None)], force_update_info=True)
print(f"Saving {out_fif.name} ...")
saved = [Path(p) for p in raw.save(out_fif, split_naming="bids", overwrite=OVERWRITE)]
for p in saved:
    print(f"Saved: {p.name} ({p.stat().st_size / 1e9:.2f} GB)")
saved_set = {p.resolve() for p in saved}
leftovers = [p for p in sorted(out_dir.glob(f"{out_base}*_meg.fif")) if p.resolve() not in saved_set]
if leftovers:
    print("These FIF files are left over from an earlier run. Delete them:", [p.name for p in leftovers])
written = []

# ---- _channels.tsv: preprocessing notebook 03's, plus OCC --------------------------------------------------
channels = pd.read_csv(meg_dir / f"{input_base}_channels.tsv", sep="\t", dtype=str, keep_default_na=False)
if add_occ:
    occ_row = {c: "n/a" for c in channels.columns}
    occ_row.update(name="OCC", type="EOG", units="T", status="good", sampling_frequency=str(SFREQ),
                   description=f"Surrogate ocular channel: {' minus '.join(surrogate_pair)} (preprocessing notebook 04, Step 4)")
    channels = pd.concat([channels, pd.DataFrame([occ_row])], ignore_index=True)
written.append(out_dir / f"{out_base}_channels.tsv")
channels.to_csv(written[-1], sep="\t", index=False)

# ---- _events.tsv / .json: preprocessing notebook 03's events, plus this notebook's labels -----------------
events_in = pd.read_csv(meg_dir / f"{input_base}_events.tsv", sep="\t", dtype=str, keep_default_na=False)
events_in = events_in[~events_in["trial_type"].isin(OUR_LABELS)]       # (none, unless re-labelling an output)
new_rows = label_rows(raw, SAVED_LABELS, events_in.columns)
written.append(out_dir / f"{out_base}_events.tsv")
sorted_events(pd.concat([events_in, new_rows], ignore_index=True), SFREQ).to_csv(written[-1], sep="\t", index=False)
events_json = read_json(meg_dir / f"{input_base}_events.json")
events_json.setdefault("trial_type", {}).setdefault("Levels", {}).update({k: LEVELS[k] for k in SAVED_LABELS})
events_json["ArtifactDetection"] = task_settings
written.append(out_dir / f"{out_base}_events.json")
write_json(written[-1], events_json)

# ---- _meg.json: preprocessing notebook 03's, with the sources (AssociatedEmptyRoom: Step 10b) ---------------
meg_json = read_json(meg_dir / f"{input_base}_meg.json")
if add_occ:
    meg_json["EOGChannelCount"] = 1
meg_json["Sources"] = [uri(DENOISE_DERIV, f) for f in denoise_fifs]
meg_json["Created"] = datetime.now().isoformat(timespec="seconds")
# Preprocessing notebook 03's link "bids::..." means "in the denoise folder"; from here it has to say so
link = meg_json.get("AssociatedEmptyRoom")
if isinstance(link, str) and link.startswith("bids::"):
    meg_json["AssociatedEmptyRoom"] = f"bids:{DENOISE_DERIV}:" + link[len("bids::"):]
written.append(out_dir / f"{out_base}_meg.json")
write_json(written[-1], meg_json)

# ---- Copied unchanged: _channels.json and the coordinate system file(s) -----------------------------------
if (meg_dir / f"{input_base}_channels.json").is_file():
    written.append(out_dir / f"{out_base}_channels.json")
    shutil.copyfile(meg_dir / f"{input_base}_channels.json", written[-1])
for coord_file in sorted(meg_dir.glob(f"{base.split('_task-')[0]}*_coordsystem.json")):
    written.append(out_dir / coord_file.name)
    shutil.copyfile(coord_file, written[-1])

# ---- Back to preprocessing notebook 03's channels and all labels in memory ---------------------------------
if add_occ:
    raw.drop_channels(["OCC"])
set_spans(raw, "OPM_blink", *opm_blink_spans)

print(f"\nSidecars written to {out_dir}:")
for p in written:
    print("   ", p.name)
print("\nLabels saved:")
display(new_rows["trial_type"].value_counts().rename("spans").to_frame())

### Step 10b: The empty room

The empty room is saved the same way, as `..._task-noise_proc-annot_meg.fif` next to the task recording, with its
labels from Step 8 (`BAD_segment`, and `BAD_jump` if turned on) in its own `_events.tsv`, and its `_meg.json`,
`_channels.tsv` and `_channels.json` copied from preprocessing notebook 03. The task recording's `_meg.json` then
gets `AssociatedEmptyRoom` pointing to it, so `bids_path.find_empty_room()` finds it from the labelled recording.

If Step 8 was skipped, the task recording's `AssociatedEmptyRoom` keeps pointing to preprocessing notebook 03's
empty room (in `derivatives/denoise/`).

In [ ]:
er_saved = []
if er is None:
    print("No empty room was checked (Step 8 skipped): AssociatedEmptyRoom points to preprocessing notebook 03's "
          f"empty room: {meg_json.get('AssociatedEmptyRoom')}")
else:
    er_in_base = er_path.copy().update(split=None, suffix=None, extension=None).basename        # ..._proc-denoise
    er_out_base = er_path.copy().update(split=None, processing=OUTPUT_PROC, suffix=None, extension=None).basename
    print(f"Saving {er_out_base}_meg.fif ...")
    er_saved = [Path(p) for p in er.save(out_dir / f"{er_out_base}_meg.fif", split_naming="bids", overwrite=OVERWRITE)]
    for p in er_saved:
        print(f"Saved: {p.name} ({p.stat().st_size / 1e9:.2f} GB)")
    er_dir = er_path.fpath.parent
    er_labels = ["BAD_jump", "BAD_segment"]

    er_written = [out_dir / f"{er_out_base}_events.tsv", out_dir / f"{er_out_base}_events.json",
                  out_dir / f"{er_out_base}_meg.json"]
    label_rows(er, er_labels, ["onset", "duration", "trial_type"]).to_csv(er_written[0], sep="\t", index=False)
    write_json(er_written[1], {
        "onset": {"Description": "Start of the span, from the first sample of the recording", "Units": "s"},
        "duration": {"Description": "Length of the span", "Units": "s"},
        "trial_type": {"Description": "Artifact label", "Levels": {k: LEVELS[k] for k in er_labels}},
        "ArtifactDetection": {"Description": "Artifact labels added by preprocessing notebook 04 (Step 8)",
                              "Channels": f"{len(er_mag_names)} good OPM magnetometer channels",
                              "JumpThresholdpT": JUMP_THRESHOLD_PT, "SegmentBandHz": list(SEGMENT_BAND_HZ),
                              "SegmentWindowSeconds": SEGMENT_WINDOW_S,
                              "SegmentChannelThreshold": EMPTY_ROOM_CHANNEL_THRESHOLD,
                              "SegmentSignificance": SEGMENT_SIGNIFICANCE, "MNEVersion": mne.__version__},
    })
    er_meg_json = read_json(er_dir / f"{er_in_base}_meg.json")
    er_meg_json["Sources"] = [uri(DENOISE_DERIV, f) for f in er.filenames]
    er_meg_json["Created"] = datetime.now().isoformat(timespec="seconds")
    write_json(er_written[2], er_meg_json)
    for suffix in ("channels.tsv", "channels.json"):
        if (er_dir / f"{er_in_base}_{suffix}").is_file():
            er_written.append(out_dir / f"{er_out_base}_{suffix}")
            shutil.copyfile(er_dir / f"{er_in_base}_{suffix}", er_written[-1])
    print(f"\nSidecars written to {out_dir}:")
    for p in er_written:
        print("   ", p.name)

    # ---- Link it from the task recording's _meg.json ("bids::" + a path inside this same dataset) ------------
    meg_json["AssociatedEmptyRoom"] = f"bids::{(sub_dir / 'meg' / er_saved[0].name).as_posix()}"
    write_json(out_dir / f"{out_base}_meg.json", meg_json)
    print(f"\nAssociatedEmptyRoom in {out_base}_meg.json:", meg_json["AssociatedEmptyRoom"])

### Step 10c: Check the saved recording

This reads the saved recording back the way the next notebook will, with `read_raw_bids`, and compares it with the
one in memory: the sampling rate, the bad channels, the `OCC` channel, and the number of spans of each label. Then it
finds the empty room from it and checks its labels too. It reads only the headers and the sidecar files, so it is
quick.

In [ ]:
check_path = mne_bids.BIDSPath(root=annot_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                               processing=OUTPUT_PROC, datatype="meg", suffix="meg", extension=".fif")
if not check_path.fpath.is_file():
    check_path.update(split="01")                  # saved in parts
check = mne_bids.read_raw_bids(check_path)         # "participants.tsv file not found": expected (see Step 2)


def count(rec, name):
    return int(np.sum(rec.annotations.description == name))


rows = [{"check": "sampling rate (Hz)", "in memory": SFREQ, "read back": check.info["sfreq"]},
        {"check": "bad channels", "in memory": sorted(raw.info["bads"]), "read back": sorted(check.info["bads"])},
        {"check": "OCC channel", "in memory": "EOG" if add_occ else "none",
         "read back": check.get_channel_types(picks=["OCC"])[0].upper() if "OCC" in check.ch_names else "none"}]
rows += [{"check": name, "in memory": count(raw, name), "read back": count(check, name)}
         for name in ["BAD_break"] + SAVED_LABELS]
if er_saved:
    er_check_path = check_path.find_empty_room(use_sidecar_only=True)
    er_check = mne_bids.read_raw_bids(er_check_path)   # "participants.tsv": expected
    rows += [{"check": f"empty room: {name}", "in memory": count(er, name), "read back": count(er_check, name)}
             for name in ("BAD_jump", "BAD_segment")]
    print("Empty room found from the saved recording:", er_check_path.basename)
results = pd.DataFrame(rows)
results["same"] = [a == b for a, b in zip(results["in memory"], results["read back"])]
display(results)
print("OPM_blink in the saved recording:", count(check, "OPM_blink"), "(should be 0: it is only used in Step 4)")

## Summary

This prints what this notebook did for this subject and session, in plain words. (A methods-section description of
the whole preprocessing is in the last preprocessing notebook.)

In [ ]:
def amount(name, rec=raw, of=None):
    s, e = get_spans(rec, name)
    of = usable_s if of is None else of
    return f"{len(s)} spans, {total_s(s, e):.1f} s ({100 * total_s(s, e) / of:.1f} %)"


summary = [f"The noise-reduced recording {input_base} (derivatives/{DENOISE_DERIV}, {SFREQ:g} Hz, "
           f"{rec_length_s(raw) / 60:.1f} min, {len(mag_names)} good OPM channels) was loaded with its sidecar files; "
           f"percentages below are of the {usable_s / 60:.1f} min outside the breaks."]
if blink_source == "eyetracker":
    summary.append(f"Blinks (BAD_blink: {amount('BAD_blink')}) were taken from the eye tracker's blinks (ET_blink), "
                   f"extended by {blink_pad[0] * 1000:.0f} ms before and {blink_pad[1] * 1000:.0f} ms after "
                   f"({'measured from the blink-locked average field' if BLINK_PAD_S == 'auto' else 'set by hand'}).")
else:
    window_from = ("set by hand" if OPM_BLINK_WINDOW_S != "auto" else
                   "from where the peaks fall in the eye tracker's blinks, plus the Step 3 pads" if opm_window_et is not None
                   else "FLUX's window")
    summary.append(f"Blinks (BAD_blink: {amount('BAD_blink')}) were found from a surrogate ocular channel "
                   f"({' minus '.join(surrogate_pair)}, {surrogate_choice}, {OPM_BLINK_BAND_HZ[0]}-{OPM_BLINK_BAND_HZ[1]} Hz, "
                   f"mne.preprocessing.find_eog_events with a threshold of {opm_thresh_T * 1e15:.0f} fT), each labelled "
                   f"from {opm_window[0]} to {opm_window[1]} s around its peak "
                   f"({window_from}); a peak within the window of the one before it was merged into that blink.")
if best_thresh_fT is not None:
    summary.append(f"Compared with the eye tracker, blinks found from the OPM sensors ({' minus '.join(surrogate_pair)}) "
                   f"matched best at a threshold of {best_thresh_fT:.0f} fT.")
if HAS_ET_SACCADES:
    summary.append(f"Saccades from the eye tracker were {'labelled BAD_saccade (' + amount('BAD_saccade') + ')' if MARK_SACCADES_BAD else 'examined but not labelled'}.")
summary.append(f"Muscle activity (BAD_muscle: {amount('BAD_muscle')}) was found as in MNE's annotate_muscle_zscore: "
               f"{MUSCLE_BAND_HZ[0]}-{MUSCLE_BAND_HZ[1]} Hz envelope"
               + (f" (with {', '.join(str(f) for f in MUSCLE_NOTCH_HZ)} Hz removed first)" if MUSCLE_NOTCH_HZ else "")
               + f", z-score threshold {MUSCLE_THRESHOLD}.")
if JUMP_THRESHOLD_PT is not None:
    summary.append(f"Sensor jumps (BAD_jump: {amount('BAD_jump')}) were found with annotate_amplitude "
                   f"(sample-to-sample change above {JUMP_THRESHOLD_PT} pT).")
summary.append(f"Other bad segments (BAD_segment: {amount('BAD_segment')}) were found as in mne-opm: GESD outliers in the "
               f"standard deviation of {SEGMENT_WINDOW_S:g} s windows of a {SEGMENT_BAND_HZ[0]}-{SEGMENT_BAND_HZ[1]} Hz copy, "
               f"a window being bad when at least {100 * SEGMENT_CHANNEL_THRESHOLD:g} % of the sensors were outliers.")
summary.append("Each step skipped the data already labelled, so no stretch carries two labels. In all, "
               f"{100 * (total_s(*bad_spans(raw)) - total_s(*break_spans)) / usable_s:.1f} % of the time outside the "
               "breaks is labelled BAD.")
if trial_loss:
    with_blinks = "; ".join(f"{d} of {n} {ev} epochs ({100 * d / n:.0f} %)" for ev, (n, d, _) in trial_loss.items())
    without = "; ".join(f"{d} of {n} {ev} epochs ({100 * d / n:.0f} %)" for ev, (n, _, d) in trial_loss.items())
    windows = "; ".join(f"{ev} {PREVIEW_EPOCHS[ev][0]:g} to {PREVIEW_EPOCHS[ev][1]:g} s" for ev in trial_loss)
    summary.append(f"For epochs of {windows}, all labels together would drop {with_blinks}. "
                   f"If the blinks are removed with ICA instead and BAD_blink is ignored, the other labels would "
                   f"drop {without}.")
if er is not None:
    summary.append(f"The empty-room recording ({er_base}) was checked the same way (BAD_segment with "
                   f"{100 * EMPTY_ROOM_CHANNEL_THRESHOLD:g} % of the sensors: "
                   f"{amount('BAD_segment', er, rec_length_s(er))} of the empty room).")
summary.append(f"The recording was saved with these labels in its _events.tsv as {', '.join(p.name for p in saved)} "
               f"(derivatives/{ANNOT_DERIV}); the data were not changed"
               + (f", and the surrogate ocular signal was added as an EOG channel, OCC ({' minus '.join(surrogate_pair)})"
                  if add_occ else "") + "."
               + (f" The empty room was saved the same way ({', '.join(p.name for p in er_saved)}) and linked from the "
                  "recording's _meg.json (AssociatedEmptyRoom)." if er_saved else ""))
print("\n\n".join(summary))